## 1. File Header, Imports, and Global Constants

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
"""

Strict context-stage patch:
- G1/G2/G3/G4/G5 are mapped to mother-cell stages 4/6/7/8/12.
- FixedTerm_OLS remains six fully independent least-squares tasks, one task per target column.
- LinearRegression/RandomForest/MLP baselines no longer receive only mother xyz; they receive
  the per-event same-CSV/same-T context: for each division event, all cells present in
  that event's own CellData CSV at that event's mother_t, their xyz coordinates,
  presence indicators, and mother identity indicators.
- Extra audit CSVs are written so lineage/event construction and baseline inputs can be checked.
"""

from __future__ import annotations

import argparse

import csv

import glob

import json

import math

import os

import re

import sys

import warnings

from pathlib import Path

from typing import Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np

import pandas as pd

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

from sklearn.exceptions import ConvergenceWarning

from sklearn.linear_model import LassoCV, LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split

from sklearn.neural_network import MLPRegressor

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

In [ ]:
try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - fallback when tqdm is unavailable.
    class tqdm:  # type: ignore
        def __init__(self, iterable=None, total=None, desc=None, unit=None, disable=False, **kwargs):
            self.iterable = iterable
            self.total = total
            self.desc = desc or ""
            self.unit = unit or "it"
            self.count = 0
            self.disable = disable
            if not self.disable and self.desc:
                print(self.desc, flush=True)

        def __iter__(self):
            for item in self.iterable:
                yield item
                self.update(1)

        def __enter__(self):
            return self

        def __exit__(self, exc_type, exc, tb):
            self.close()

        def update(self, n=1):
            self.count += n
            if not self.disable and self.total and (self.count == self.total or self.count % max(1, self.total // 20) == 0):
                print(f"{self.desc}: {self.count}/{self.total} {self.unit}", flush=True)

        def set_description_str(self, desc):
            self.desc = desc
            if not self.disable:
                print(desc, flush=True)

        def close(self):
            pass

        @staticmethod
        def write(msg):
            print(msg, flush=True)

try:
    from cell_pipeline.features import CellFeatureExtractor
except Exception as exc:  # pragma: no cover - this is a user-facing runtime guard.
    raise ImportError(
        "Failed to import cell_pipeline.features.CellFeatureExtractor."
        "Run this script under the original project's analysis/ directory, or make sure analysis/ is in PYTHONPATH."
    ) from exc

# Raw daughter-pair targets used for FixedTerm_OLS fitting and final metrics.
TARGET_COLS = [
    "x_mean",
    "x_half_absdiff",
    "y_mean",
    "y_half_absdiff",
    "z_mean",
    "z_half_absdiff",
]

TARGET_DISPLAY = {
    "x_mean": "(x1+x2)/2",
    "x_half_absdiff": "|x1-x2|/2",
    "y_mean": "(y1+y2)/2",
    "y_half_absdiff": "|y1-y2|/2",
    "z_mean": "(z1+z2)/2",
    "z_half_absdiff": "|z1-z2|/2",
}

# Same columns are used for evaluation.  Baselines internally fit residuals for
# mean targets but return reconstructed raw predictions on these columns.
EVAL_TARGET_COLS = list(TARGET_COLS)

EVAL_TARGET_DISPLAY = dict(TARGET_DISPLAY)

# Baseline-only residual targets for the mean part.
BASELINE_MODEL_TARGET_COLS = [
    "x_mean_minus_mother_x",
    "x_half_absdiff",
    "y_mean_minus_mother_y",
    "y_half_absdiff",
    "z_mean_minus_mother_z",
    "z_half_absdiff",
]

In [ ]:
MEAN_RESIDUAL_SPECS = [
    {"axis": "x", "mean_col": "x_mean", "mother_col": "mother_x", "delta_col": "delta_x_mean", "target_index": 0},
    {"axis": "y", "mean_col": "y_mean", "mother_col": "mother_y", "delta_col": "delta_y_mean", "target_index": 2},
    {"axis": "z", "mean_col": "z_mean", "mother_col": "mother_z", "delta_col": "delta_z_mean", "target_index": 4},
]

XYZ_COLS = ["mother_x", "mother_y", "mother_z"]

EVENT_META_COLS = [
    "sample_id", "dataset_label", "file_name", "file_path", "group_idx",
    "transition", "mother_t", "daughter_t", "mother_stage_count", "daughter_stage_count",
    "lineage_rule", "mother_name", "daughter1_name", "daughter2_name",
]

FEATURE_META_COLS = EVENT_META_COLS + XYZ_COLS + [
    "mother_cell_stage_count",
    "adjacency_stage_status",
]

# Early lineage relations used by the datasets mentioned in the prompt.
# These rules are name-based mother -> daughter relations.  The biological
# stage used for adjacency is NOT hard-coded here; it is inferred from the
# number of cells present at the mother time point and then matched through
# G_TO_CELL_STAGE below.
DEFAULT_LINEAGE_MAP: Dict[str, Tuple[str, str]] = {
    "ABa": ("ABal", "ABar"),
    "ABp": ("ABpl", "ABpr"),
    "EMS": ("MS", "E"),
    "P2": ("C", "P3"),
    "ABal": ("ABala", "ABalp"),
    "ABar": ("ABara", "ABarp"),
    "ABpl": ("ABpla", "ABplp"),
    "ABpr": ("ABpra", "ABprp"),
    "MS": ("MSa", "MSp"),
    "E": ("Ea", "Ep"),
}

DEFAULT_FEATURE_TOGGLES = {
    "self_single_dim": True,
    "self_coupled_polynomial": True,
    "adjacent_polynomial": True,
    "self_other_functions": False,
    "adjacent_other_functions": False,
}

## 2. Argument Parsers and Basic CSV Reading

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def parse_label_path(text: str, arg_name: str = "argument") -> Tuple[str, str]:
    if "=" not in text:
        raise argparse.ArgumentTypeError(f"{arg_name} must have the form LABEL=PATH; received: {text}")
    label, path = text.split("=", 1)
    label = label.strip()
    path = path.strip()
    if not label or not path:
        raise argparse.ArgumentTypeError(f"Invalid {arg_name}: {text}")
    return label, path

def parse_eval_cell_data(text: str) -> Tuple[str, str]:
    return parse_label_path(text, "--eval-cell-data")

def parse_eval_adjacency(text: str) -> Tuple[str, str]:
    return parse_label_path(text, "--eval-adjacency")

def parse_train_fracs(text: str) -> List[float]:
    vals = [float(x.strip()) for x in text.split(",") if x.strip()]
    if not vals:
        raise argparse.ArgumentTypeError("--train-fracs cannot be empty")
    for v in vals:
        if not (0.0 < v < 1.0):
            raise argparse.ArgumentTypeError(f"train fraction must be in (0,1); received {v}")
    return vals

def parse_train_sizes(text: str) -> List[int]:
    """Parse train sizes such as "5,10,20,30", "5 10 20", or "20-100:10".

    Plain ranges like "1-100" are still supported for compatibility and mean
    every integer.  For step ranges, use "20-100:10".
    """
    vals: List[int] = []
    parts = re.split(r"[\s,]+", str(text).strip())
    for part in parts:
        part = part.strip()
        if not part:
            continue
        step = 1
        range_part = part
        if ":" in part:
            range_part, step_text = part.split(":", 1)
            step = int(step_text.strip())
            if step <= 0:
                raise argparse.ArgumentTypeError(f"Invalid train size step: {part}")
        if "-" in range_part:
            a, b = [int(x.strip()) for x in range_part.split("-", 1)]
            if a <= 0 or b <= 0 or b < a:
                raise argparse.ArgumentTypeError(f"Invalid train size range: {part}")
            vals.extend(range(a, b + 1, step))
        else:
            v = int(range_part)
            if v <= 0:
                raise argparse.ArgumentTypeError(f"train size must be a positive integer; received {v}")
            vals.append(v)
    vals = sorted(set(vals))
    if not vals:
        raise argparse.ArgumentTypeError("--train-sizes cannot be empty")
    return vals

def parse_feature_toggles(text: Optional[str]) -> Dict[str, bool]:
    toggles = dict(DEFAULT_FEATURE_TOGGLES)
    if not text:
        return toggles
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        if "=" not in part:
            raise argparse.ArgumentTypeError(
                "--feature-toggles must use key=true/false format, for example adjacent_polynomial=true"
            )
        key, val = [x.strip() for x in part.split("=", 1)]
        if key not in toggles:
            raise argparse.ArgumentTypeError(f"Unknown feature toggle: {key}; options {sorted(toggles)}")
        toggles[key] = val.lower() in {"1", "true", "yes", "y", "on"}
    return toggles

In [ ]:
def discover_cell_data_files(cell_data_input: str) -> List[str]:
    p = Path(cell_data_input)
    if p.is_file():
        return [str(p)]
    if p.is_dir():
        files = sorted(glob.glob(str(p / "CellData_*.csv")))
        if not files:
            files = sorted(glob.glob(str(p / "*.csv")))
        return files
    return sorted(glob.glob(cell_data_input))

def infer_group_idx_from_path(path: str) -> int:
    name = os.path.basename(path)
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else 0

def clean_cell_name(x: object) -> str:
    return str(x).replace("'", "").replace('"', "").strip()

def load_cell_data_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    required = {"CellName", "T", "X", "Y", "Z"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    df = df[["CellName", "T", "X", "Y", "Z"]].copy()
    df["CellName"] = df["CellName"].map(clean_cell_name)
    df["T"] = pd.to_numeric(df["T"], errors="coerce").astype("Int64")
    for c in ["X", "Y", "Z"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["CellName", "T", "X", "Y", "Z"]).copy()
    df["T"] = df["T"].astype(int)
    return df.reset_index(drop=True)

def _row_coords(row: pd.Series) -> Tuple[float, float, float]:
    return float(row["X"]), float(row["Y"]), float(row["Z"])

## 3. Constructing Mother-to-Daughter Division Events from CellData

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def build_events_from_cell_file(
    path: str,
    dataset_label: str,
    lineage_map: Mapping[str, Tuple[str, str]],
) -> pd.DataFrame:
    df = load_cell_data_file(path)
    group_idx = infer_group_idx_from_path(path)
    file_name = os.path.basename(path)

    rows: List[Dict[str, object]] = []
    times = sorted(df["T"].unique().tolist())
    time_to_rows: Dict[int, Dict[str, pd.Series]] = {}
    for t in times:
        t_df = df[df["T"] == t].copy()
        time_to_rows[int(t)] = {str(r["CellName"]): r for _, r in t_df.iterrows()}

    for t in times:
        t_next = int(t) + 1
        if t_next not in time_to_rows:
            continue
        mothers = time_to_rows[int(t)]
        next_cells = time_to_rows[t_next]
        for mother, daughters in lineage_map.items():
            d1, d2 = daughters
            if mother not in mothers or d1 not in next_cells or d2 not in next_cells:
                continue
            m = mothers[mother]
            c1 = next_cells[d1]
            c2 = next_cells[d2]
            mx, my, mz = _row_coords(m)
            x1, y1, z1 = _row_coords(c1)
            x2, y2, z2 = _row_coords(c2)

            x_mean = (x1 + x2) / 2.0
            y_mean = (y1 + y2) / 2.0
            z_mean = (z1 + z2) / 2.0

            rows.append({
                "sample_id": f"{Path(path).stem}|T{int(t)}-{t_next}|{mother}",
                "dataset_label": dataset_label,
                "file_name": file_name,
                "file_path": path,
                "group_idx": group_idx,
                "transition": f"T{int(t)}-{t_next}",
                "mother_t": int(t),
                "daughter_t": t_next,
                "mother_stage_count": int(len(mothers)),
                "daughter_stage_count": int(len(next_cells)),
                "lineage_rule": f"{mother}->{d1},{d2}",
                "mother_name": mother,
                "daughter1_name": d1,
                "daughter2_name": d2,
                "mother_x": mx,
                "mother_y": my,
                "mother_z": mz,
                "daughter1_x": x1,
                "daughter1_y": y1,
                "daughter1_z": z1,
                "daughter2_x": x2,
                "daughter2_y": y2,
                "daughter2_z": z2,
                # Raw daughter-pair means.  In v9 these are also the final
                # evaluation targets after reconstructing pred_delta + c*mother.
                "x_mean": x_mean,
                "y_mean": y_mean,
                "z_mean": z_mean,
                # Placeholder residual targets.  These are overwritten inside
                # every train/test split using c fitted on the training data only.
                "delta_x_mean": x_mean - mx,
                "x_half_absdiff": abs(x1 - x2) / 2.0,
                "delta_y_mean": y_mean - my,
                "y_half_absdiff": abs(y1 - y2) / 2.0,
                "delta_z_mean": z_mean - mz,
                "z_half_absdiff": abs(z1 - z2) / 2.0,
            })

    return pd.DataFrame(rows)

In [ ]:
def build_dataset_from_cell_data(
    cell_data_input: str,
    dataset_label: str,
    lineage_map: Mapping[str, Tuple[str, str]],
) -> pd.DataFrame:
    files = discover_cell_data_files(cell_data_input)
    if not files:
        raise FileNotFoundError(f"No CellData files found: {cell_data_input}")
    dfs = []
    for path in files:
        events = build_events_from_cell_file(path, dataset_label=dataset_label, lineage_map=lineage_map)
        if not events.empty:
            dfs.append(events)
    if not dfs:
        raise ValueError(
            f"{dataset_label} did not produce any mother-daughter division events; check CellName/T or lineage_map. Input: {cell_data_input}"
        )
    out = pd.concat(dfs, ignore_index=True)
    out = out.sort_values(["file_name", "transition", "mother_name"]).reset_index(drop=True)
    return out

def load_lineage_json(path: Optional[str]) -> Dict[str, Tuple[str, str]]:
    if not path:
        return dict(DEFAULT_LINEAGE_MAP)
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    lineage: Dict[str, Tuple[str, str]] = {}
    for mother, daughters in raw.items():
        if not isinstance(daughters, (list, tuple)) or len(daughters) != 2:
            raise ValueError(f"In lineage JSON, {mother} must have daughters as a list of length 2")
        lineage[str(mother)] = (str(daughters[0]), str(daughters[1]))
    return lineage

## 4. Gi Adjacency Matrices and the FixedTerm Feature Library

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def load_adjacency_csv(path: str) -> Dict[str, Dict[str, float]]:
    """Read a square cell adjacency CSV into {cell: {neighbor: strength}}.

    The reader is intentionally tolerant of quoted headers such as 'ABal', of a
    first column named Cell Identity / CellName, and of missing or non-numeric
    entries. Values are clipped to [0, 1].
    """
    df = pd.read_csv(path, dtype=str)
    if df.empty:
        return {}
    df.columns = [clean_cell_name(c) for c in df.columns]
    first_col = df.columns[0]

    # Most adjacency files have the first column as the row cell name.
    first_col_values = df[first_col].map(clean_cell_name)
    first_col_numeric = pd.to_numeric(first_col_values, errors="coerce").notna().mean()
    first_col_is_cell_id = first_col.lower() in {
        "cell identity", "cell_identity", "cellname", "cell", "name", "unnamed: 0", ""
    }
    if first_col_is_cell_id or first_col_numeric < 0.8:
        df[first_col] = first_col_values
        df = df.set_index(first_col)
    else:
        # Fall back to using columns as both row/col identities when no row-name
        # column exists. This is uncommon but keeps the parser robust.
        df.index = [clean_cell_name(x) for x in df.index]

    df.index = [clean_cell_name(x) for x in df.index]
    df.columns = [clean_cell_name(c) for c in df.columns]

    for c in df.columns:
        df[c] = df[c].map(clean_cell_name)
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    result: Dict[str, Dict[str, float]] = {}
    for cell in df.index:
        row: Dict[str, float] = {}
        for nb in df.columns:
            if nb == cell:
                continue
            val = float(df.loc[cell, nb])
            val = max(0.0, min(1.0, val))
            if val > 0:
                row[nb] = val
        result[str(cell)] = row
    return result

# Your adjacency files are named by *mother-cell stage*, not by the local T column.
# In this project the five Gi.csv files correspond exactly to the number of cells
# simultaneously present at the mother time point:
#   adj/G1.csv -> 4-cell mother stage
#   adj/G2.csv -> 6-cell mother stage
#   adj/G3.csv -> 7-cell mother stage
#   adj/G4.csv -> 8-cell mother stage
#   adj/G5.csv -> 12-cell mother stage
G_TO_CELL_STAGE = {1: 4, 2: 6, 3: 7, 4: 8, 5: 12}

In [ ]:
def discover_adjacency_files(adjacency_input: Optional[str]) -> Dict[int, Dict[str, Dict[str, float]]]:
    """Return {cell_stage_count: adjacency_dict}.

    This function intentionally does NOT map Gk to local time T=k-1.  In the
    user's data, G1/G2/G3/G4/G5 denote the biological cell stages
    4/6/7/8/12.  For example, in an 8-cell-stage mother time point the
    mother cells may live at local T=0, but their stage is 8 cells, so they
    must use adj/G4.csv rather than adj/G1.csv.

    Supported inputs:
    - None: no adjacency, adjacent feature values become zero;
    - adj/ directory: reads G1.csv ... G5.csv and maps them to stages
      4, 6, 7, 8, 12;
    - a single Gk.csv file: maps to its corresponding stage;
    - a non-G single CSV: stored as fallback key -1, used only if no exact
      stage-specific matrix is available.
    """
    if not adjacency_input:
        return {}
    p = Path(adjacency_input)
    if p.is_file():
        files = [str(p)]
    elif p.is_dir():
        files = sorted(glob.glob(str(p / "G*.csv")))
        if not files:
            files = sorted(glob.glob(str(p / "*.csv")))
    else:
        files = sorted(glob.glob(adjacency_input))
    if not files:
        raise FileNotFoundError(f"No adjacency CSV files found: {adjacency_input}")

    out: Dict[int, Dict[str, Dict[str, float]]] = {}
    for path in files:
        name = os.path.basename(path)
        m = re.search(r"G(\d+)", name, re.I)
        if m:
            g_idx = int(m.group(1))
            if g_idx not in G_TO_CELL_STAGE:
                raise ValueError(f"Unknown adjacency matrix index {name}: currently only G1-G5 are supported, corresponding to mother stages 4/6/7/8/12")
            stage_count = G_TO_CELL_STAGE[g_idx]
        elif len(files) == 1:
            # Backward-compatible fallback for an explicitly supplied CSV such as
            # --eval-adjacency LABEL=cell8.csv.  Prefer adj/G*.csv style.
            stage_count = -1
        else:
            # Avoid guessing for multi-file non-G directories.
            continue
        out[stage_count] = load_adjacency_csv(path)
    return out

def binarize_adjacency_by_stage(
    adjacency_by_stage: Mapping[int, Dict[str, Dict[str, float]]],
) -> Dict[int, Dict[str, Dict[str, float]]]:
    """Return a copy where every positive contact strength is replaced by 1.

    This implements the second FixedTerm variant: the support/nonzero contact
    graph from Gi.csv is kept, but all positive contact strengths are treated as
    equal weights.  The mapping G1/G2/G3/G4/G5 -> 4/6/7/8/12 is unchanged.
    """
    out: Dict[int, Dict[str, Dict[str, float]]] = {}
    for stage, adj in adjacency_by_stage.items():
        out[int(stage)] = {
            str(cell): {str(nb): 1.0 for nb, val in nbs.items() if float(val) > 0.0}
            for cell, nbs in adj.items()
        }
    return out

In [ ]:
def build_timepoint_cells(
    cell_df: pd.DataFrame,
    adjacency_by_stage: Mapping[int, Dict[str, Dict[str, float]]],
) -> Dict[int, Dict[str, Dict[str, object]]]:
    timepoint_cells: Dict[int, Dict[str, Dict[str, object]]] = {}
    for t, t_df in cell_df.groupby("T", sort=True):
        t = int(t)
        cells_at_t = set(t_df["CellName"].astype(str).tolist())
        stage_count = len(cells_at_t)
        # Use the adjacency matrix whose biological stage matches the number of
        # cells at this time point.  This is the key change: local T=0 in an
        # For a file whose mother time point has 8 cells, use stage_count=8 -> G4.csv.
        if stage_count in adjacency_by_stage:
            adj_t = adjacency_by_stage[stage_count]
            adjacency_stage_status = f"exact_stage_{stage_count}"
        elif -1 in adjacency_by_stage:
            # Backward-compatible single-file fallback, e.g. a legacy cell8.csv.
            adj_t = adjacency_by_stage[-1]
            adjacency_stage_status = "single_csv_fallback"
        else:
            adj_t = {}
            adjacency_stage_status = "missing"
        current: Dict[str, Dict[str, object]] = {}
        for _, row in t_df.iterrows():
            cell_name = str(row["CellName"])
            raw_adj = adj_t.get(cell_name, {})
            filtered_adj = {
                nb: float(v)
                for nb, v in raw_adj.items()
                if nb in cells_at_t and nb != cell_name and float(v) > 0
            }
            current[cell_name] = {
                "self_coords": np.array([row["X"], row["Y"], row["Z"]], dtype=float),
                "adjacency_matrix": filtered_adj,
                "gi_group": f"G{t+1}",
                "cell_stage_count": stage_count,
                "adjacency_stage_status": adjacency_stage_status,
            }
        timepoint_cells[t] = current
    return timepoint_cells

def compute_feature_vector_for_cell(
    extractor: CellFeatureExtractor,
    current_cell_name: str,
    current_cell_coords: np.ndarray,
    current_time_cells: Dict[str, Dict[str, object]],
    adj_matrix: Dict[str, float],
    prefix: str = "mother_",
) -> Tuple[List[float], List[str]]:
    """Compute exactly the feature-library row used by FixedTerm_OLS."""
    s_feats, s_names = extractor._self_single_dim_features(current_cell_coords, prefix=prefix)
    cp_feats, cp_names = extractor._self_coupled_polynomial_features(current_cell_coords, prefix=prefix)
    o_feats, o_names = extractor._self_other_functions_features(current_cell_coords, prefix=prefix)
    adj_feats, adj_names = extractor._sum_with_adj_cells(
        current_cell_name=current_cell_name,
        current_cell_coords=current_cell_coords,
        current_time_cells=current_time_cells,
        adj_matrix=adj_matrix,
        prefix=prefix,
    )
    feats = s_feats + cp_feats + o_feats + adj_feats
    names = s_names + cp_names + o_names + adj_names
    clean_feats = []
    for v in feats:
        try:
            vv = float(v)
        except Exception:
            vv = 0.0
        if not np.isfinite(vv):
            vv = 0.0
        clean_feats.append(vv)
    return clean_feats, names

In [ ]:
def build_feature_library_for_events(
    events: pd.DataFrame,
    adjacency_by_t: Mapping[int, Dict[str, Dict[str, float]]],
    feature_toggles: Mapping[str, bool],
    prefix: str = "mother_",
) -> pd.DataFrame:
    """Build one CellFeatureExtractor feature row per event, aligned with events."""
    extractor = CellFeatureExtractor(feature_toggles=dict(feature_toggles))
    rows: List[Dict[str, object]] = []
    cache: Dict[str, Dict[int, Dict[str, Dict[str, object]]]] = {}

    expected_names: Optional[List[str]] = None
    for _, event in events.iterrows():
        file_path = str(event["file_path"])
        if file_path not in cache:
            df = load_cell_data_file(file_path)
            cache[file_path] = build_timepoint_cells(df, adjacency_by_t)
        t = int(event["mother_t"])
        mother_name = str(event["mother_name"])
        if t not in cache[file_path] or mother_name not in cache[file_path][t]:
            raise KeyError(f"Unable to find {file_path} at T={t} mother cell {mother_name}")
        current_time_cells = cache[file_path][t]
        mother_data = current_time_cells[mother_name]
        coords = np.asarray(mother_data["self_coords"], dtype=float)
        adj_matrix = dict(mother_data.get("adjacency_matrix", {}))

        feats, names = compute_feature_vector_for_cell(
            extractor=extractor,
            current_cell_name=mother_name,
            current_cell_coords=coords,
            current_time_cells=current_time_cells,
            adj_matrix=adj_matrix,
            prefix=prefix,
        )
        if expected_names is None:
            expected_names = list(names)
        elif expected_names != list(names):
            raise ValueError(
                "Feature column names differ across events; check feature_toggles or CellFeatureExtractor."
            )

        row = {col: event[col] for col in EVENT_META_COLS if col in event.index}
        row.update({
            "mother_x": float(coords[0]),
            "mother_y": float(coords[1]),
            "mother_z": float(coords[2]),
            "mother_cell_stage_count": int(mother_data.get("cell_stage_count", -1)),
            "adjacency_stage_status": str(mother_data.get("adjacency_stage_status", "missing")),
        })
        row.update(dict(zip(names, feats)))
        rows.append(row)

    feature_df = pd.DataFrame(rows)
    value_cols = [c for c in feature_df.columns if c not in FEATURE_META_COLS]
    feature_df[value_cols] = feature_df[value_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return feature_df

## 5. Audit Tables, same-CSV/same-T Context, and the FixedTerm Context Variant

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def audit_event_table(events: pd.DataFrame, dataset_label: str = "") -> pd.DataFrame:
    """Summarize constructed mother -> two-daughter events for manual verification.

    This audit is deliberately name- and stage-explicit.  If data were read from
    a wrong file/time point, the mother_stage_count/daughter_stage_count and the
    mother/daughter names in this table should make the problem visible.
    """
    if events.empty:
        return pd.DataFrame()
    group_cols = [
        "dataset_label", "transition", "mother_stage_count", "daughter_stage_count",
        "mother_name", "daughter1_name", "daughter2_name", "lineage_rule",
    ]
    group_cols = [c for c in group_cols if c in events.columns]
    audit = (
        events.groupby(group_cols, dropna=False)
        .agg(
            n_events=("sample_id", "count"),
            n_files=("file_name", "nunique"),
            first_file=("file_name", "min"),
            last_file=("file_name", "max"),
        )
        .reset_index()
    )
    if dataset_label:
        audit["audit_source"] = dataset_label
    return audit

def _all_context_cell_names(events: pd.DataFrame) -> List[str]:
    """Column universe for baseline context features.

    Important: this helper is used ONLY to make a fixed-width design matrix for
    sklearn.  For each individual event, the values are still read strictly from
    that event's own `file_path` and that event's own `mother_t`.  Therefore an
    ABa division in one CellData_*.csv at a 4-cell mother stage uses the cells
    present in that same CSV and that same T, e.g. ABa/ABp/EMS/P2.
    """
    names = set()
    cache: Dict[str, pd.DataFrame] = {}
    for _, event in events.iterrows():
        file_path = str(event["file_path"])
        if file_path not in cache:
            cache[file_path] = load_cell_data_file(file_path)
        df = cache[file_path]
        t = int(event["mother_t"])
        part = df[df["T"].astype(int) == t]
        names.update(part["CellName"].astype(str).tolist())
    return sorted(names)

In [ ]:
def build_same_time_cell_context_features(events: pd.DataFrame) -> pd.DataFrame:
    """Build baseline inputs from the same CSV and same T as each event.

    This is the baseline input rule requested here:
      for one division event, go back to its own CellData CSV (`file_path`), take
      exactly the rows with `T == mother_t`, and feed the coordinates of ALL cells
      present at that time point.  For example, when ABa divides at the 4-cell
      stage, the row is built from ABa, ABp, EMS, and P2 in that same CSV and
      that same T.

    Because sklearn needs fixed-width matrices, columns are created for the union
    of cell names seen in the supplied events, but for each event only cells
    present in its own same-CSV/same-T snapshot have present=1 and nonzero xyz.
    The output includes `context_cells_present` as an explicit audit field.
    Daughter coordinates are never used as baseline inputs.
    """
    context_cell_names = _all_context_cell_names(events)
    rows: List[Dict[str, object]] = []
    cache: Dict[str, pd.DataFrame] = {}
    for _, event in events.iterrows():
        file_path = str(event["file_path"])
        if file_path not in cache:
            cache[file_path] = load_cell_data_file(file_path)
        df = cache[file_path]
        t = int(event["mother_t"])
        mother_name = str(event["mother_name"])
        # The crucial line: the baseline context is exactly same CSV + same T.
        part = df[df["T"].astype(int) == t].copy()
        if part.empty:
            raise ValueError(f"baseline context is empty: file={file_path}, T={t}, mother={mother_name}")
        cell_names_this_event = part["CellName"].astype(str).tolist()
        if mother_name not in set(cell_names_this_event):
            raise ValueError(
                f"division mother cell not found in baseline context: file={file_path}, T={t}, "
                f"mother={mother_name}, context_cells={cell_names_this_event}"
            )
        cell_to_row = {str(r["CellName"]): r for _, r in part.iterrows()}
        row: Dict[str, object] = {
            "sample_id": event.get("sample_id", ""),
            "file_name": event.get("file_name", ""),
            "file_path": file_path,
            "mother_t": t,
            "mother_stage_count": int(event.get("mother_stage_count", len(part))),
            "mother_name": mother_name,
            "context_source_rule": "same_file_same_T_all_cells",
            "context_cell_count": int(len(cell_names_this_event)),
            "context_cells_present": ";".join(cell_names_this_event),
        }
        for cell in context_cell_names:
            present = cell in cell_to_row
            row[f"ctx_{cell}_present"] = 1.0 if present else 0.0
            # This tells the baseline which of the same-time cells is the mother
            # for the current prediction task. Without it, multiple dividing
            # mothers from the same CSV/T would have identical inputs but
            # different targets.
            row[f"ctx_{cell}_is_mother"] = 1.0 if (present and cell == mother_name) else 0.0
            if present:
                r = cell_to_row[cell]
                row[f"ctx_{cell}_x"] = float(r["X"])
                row[f"ctx_{cell}_y"] = float(r["Y"])
                row[f"ctx_{cell}_z"] = float(r["Z"])
            else:
                row[f"ctx_{cell}_x"] = 0.0
                row[f"ctx_{cell}_y"] = 0.0
                row[f"ctx_{cell}_z"] = 0.0
        rows.append(row)
    out = pd.DataFrame(rows)
    value_cols = [c for c in out.columns if c.startswith("ctx_")]
    out[value_cols] = out[value_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out

def baseline_context_value_columns(context_df: pd.DataFrame) -> List[str]:
    return [c for c in context_df.columns if c.startswith("ctx_")]

In [ ]:
def append_same_time_context_to_fixedterm_features(
    feature_df: pd.DataFrame,
    events: pd.DataFrame,
    prefix: str = "ftctx_",
) -> pd.DataFrame:
    """Append same-CSV/same-T all-current-cell context to a FixedTerm feature table.

    This is used only by ``FixedTerm_OLS_binaryAdj_context``.  For each event,
    the added values are built from that event's own ``file_path`` and
    ``mother_t``: all rows in the same CellData CSV with ``T == mother_t``.
    Thus an ABa division at the 4-cell stage uses ABa/ABp/EMS/P2 from that
    same CSV and same T.  Daughter coordinates at T+1 are never used.

    The baseline context columns are renamed from ``ctx_*`` to ``ftctx_*`` so
    that audit tables can distinguish these FixedTerm-context regressors from
    baseline-only context regressors.
    """
    if len(feature_df) != len(events):
        raise ValueError(
            f"feature_df and events have different row counts: len(feature_df)={len(feature_df)}, len(events)={len(events)}"
        )

    base = feature_df.reset_index(drop=True).copy()
    context = build_same_time_cell_context_features(events.reset_index(drop=True))
    ctx_cols = baseline_context_value_columns(context)
    if not ctx_cols:
        raise ValueError("same-time context did not generate any numeric ctx_* columns, so it cannot be appended to FixedTerm features.")

    context_numeric = context[ctx_cols].copy()
    context_numeric.columns = [f"{prefix}{c[4:] if c.startswith('ctx_') else c}" for c in ctx_cols]
    context_numeric = context_numeric.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    out = pd.concat([base, context_numeric], axis=1)
    return out

def fixedterm_context_value_columns(feature_df: pd.DataFrame, prefix: str = "ftctx_") -> List[str]:
    """Return the same-time context regressors appended to a FixedTerm feature table."""
    return [c for c in feature_df.columns if str(c).startswith(prefix)]

def add_context_terms_to_selected_terms(
    selected_terms: Mapping[str, List[str]],
    feature_df_with_context: pd.DataFrame,
    prefix: str = "ftctx_",
) -> Dict[str, List[str]]:
    """Append all same-time context terms to every independent target's term list.

    This keeps the fitting principle unchanged: each target column is still an
    independent OLS problem.  The only change is that the design matrix for the
    ``*_context`` variant contains the original selected FixedTerm columns plus
    all ``ftctx_*`` same-CSV/same-T context columns.
    """
    context_terms = fixedterm_context_value_columns(feature_df_with_context, prefix=prefix)
    if not context_terms:
        raise ValueError("FixedTerm context variant did not find any ftctx_* feature columns.")

    resolved: Dict[str, List[str]] = {}
    for target in TARGET_COLS:
        base_terms = list(selected_terms.get(target, []))
        if not base_terms:
            raise KeyError(f"target={target} has no base fixed terms, so context terms cannot be appended.")
        merged: List[str] = []
        for term in base_terms + context_terms:
            if term in feature_df_with_context.columns and term not in merged:
                merged.append(term)
        resolved[target] = merged
    return resolved

def feature_value_columns(feature_df: pd.DataFrame) -> List[str]:
    return [c for c in feature_df.columns if c not in FEATURE_META_COLS]

def fixed_term_candidate_columns(feature_df: pd.DataFrame) -> List[str]:
    """Columns allowed as fixed-term regressors.

    Unlike feature_value_columns(), this intentionally keeps mother_x/mother_y/
    mother_z, because regression_equations.csv may contain simple terms x,y,z,
    which canonicalize to mother_x/mother_y/mother_z.
    """
    forbidden = set(EVENT_META_COLS) | {"mother_cell_stage_count", "adjacency_stage_status"}
    return [c for c in feature_df.columns if c not in forbidden]

## 6. Basic Metrics and Geometric Errors

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanvar(y_true) <= 1e-14:
        return float("nan")
    return float(r2_score(y_true, y_pred))

def safe_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanstd(y_true) <= 1e-14 or np.nanstd(y_pred) <= 1e-14:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def safe_spearman(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2:
        return float("nan")
    r1 = pd.Series(np.asarray(y_true, dtype=float)).rank(method="average").to_numpy(dtype=float)
    r2 = pd.Series(np.asarray(y_pred, dtype=float)).rank(method="average").to_numpy(dtype=float)
    return safe_corr(r1, r2)

def safe_explained_variance(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanvar(y_true) <= 1e-14:
        return float("nan")
    return float(1.0 - np.nanvar(y_true - y_pred) / np.nanvar(y_true))

def _safe_div(num: float, den: float) -> float:
    return float(num / den) if np.isfinite(num) and np.isfinite(den) and abs(den) > 1e-14 else float("nan")

METRIC_KEYS = ["r2", "rmse", "mae"]

def compute_metric_dict(yt: np.ndarray, yp: np.ndarray) -> Dict[str, float]:
    """Core scalar metrics only: R2, RMSE, MAE."""
    yt = np.asarray(yt, dtype=float)
    yp = np.asarray(yp, dtype=float)
    if len(yt) == 0:
        return {k: float("nan") for k in METRIC_KEYS}
    mse = float(mean_squared_error(yt, yp))
    return {
        "r2": safe_r2(yt, yp),
        "rmse": float(math.sqrt(mse)) if np.isfinite(mse) else float("nan"),
        "mae": float(mean_absolute_error(yt, yp)),
    }

In [ ]:
def metric_rows(
    y_true: pd.DataFrame,
    y_pred: np.ndarray,
    model_name: str,
    dataset_label: str,
    train_size: int,
    repeat: int,
    n_train: int,
    n_test: int,
    split_meta: Optional[Mapping[str, object]] = None,
) -> List[Dict[str, object]]:
    split_meta = dict(split_meta or {})
    rows: List[Dict[str, object]] = []
    for j, target in enumerate(EVAL_TARGET_COLS):
        yt = y_true[target].to_numpy(dtype=float)
        yp = np.asarray(y_pred[:, j], dtype=float)
        row = {
            "dataset_label": dataset_label,
            "train_size": int(train_size),
            "train_fraction_actual": float(n_train / (n_train + n_test)) if (n_train + n_test) else float("nan"),
            "repeat": repeat,
            "model_name": model_name,
            "target": target,
            "target_display": EVAL_TARGET_DISPLAY[target],
            "target_group": "mean_position" if target in {"x_mean", "y_mean", "z_mean"} else "half_absdiff",
            "n_train": n_train,
            "n_test": n_test,
            "split_unit_actual": split_meta.get("split_unit_actual", ""),
            "split_group_col": split_meta.get("split_group_col", ""),
            "n_train_groups": split_meta.get("n_train_groups", np.nan),
            "n_test_groups": split_meta.get("n_test_groups", np.nan),
        }
        row.update(compute_metric_dict(yt, yp))
        rows.append(row)

    def add_macro(name: str, display: str, group: str, selected_rows: List[Dict[str, object]]) -> None:
        macro = {
            "dataset_label": dataset_label,
            "train_size": int(train_size),
            "train_fraction_actual": float(n_train / (n_train + n_test)) if (n_train + n_test) else float("nan"),
            "repeat": repeat,
            "model_name": model_name,
            "target": name,
            "target_display": display,
            "target_group": group,
            "n_train": n_train,
            "n_test": n_test,
            "split_unit_actual": split_meta.get("split_unit_actual", ""),
            "split_group_col": split_meta.get("split_group_col", ""),
            "n_train_groups": split_meta.get("n_train_groups", np.nan),
            "n_test_groups": split_meta.get("n_test_groups", np.nan),
        }
        for key in METRIC_KEYS:
            vals = np.array([r[key] for r in selected_rows], dtype=float)
            macro[key] = float(np.nanmean(vals)) if np.isfinite(vals).any() else float("nan")
        rows.append(macro)

    add_macro("macro_mean", "macro mean over all 6 raw targets", "all", rows[:])
    add_macro(
        "macro_mean_position",
        "macro mean over x_mean/y_mean/z_mean",
        "mean_position",
        [r for r in rows if r["target"] in {"x_mean", "y_mean", "z_mean"}],
    )
    add_macro(
        "macro_half_absdiff",
        "macro mean over x/y/z half_absdiff",
        "half_absdiff",
        [r for r in rows if r["target"] in {"x_half_absdiff", "y_half_absdiff", "z_half_absdiff"}],
    )

    return rows

GEOMETRIC_ERROR_KEYS = ["split_vector_error"]

GEOMETRIC_EXTRA_KEYS: List[str] = []

SKILL_REFERENCE_MODELS: List[str] = []

def _norm_rows(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"expected 2D array, got shape={a.shape}")
    return np.linalg.norm(a, axis=1)

In [ ]:
def _angle_deg_rows(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Row-wise angle in degrees. Returns NaN when one vector is zero."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    an = _norm_rows(a)
    bn = _norm_rows(b)
    denom = an * bn
    out = np.full(len(a), np.nan, dtype=float)
    mask = denom > 1e-14
    if mask.any():
        cosv = np.sum(a[mask] * b[mask], axis=1) / denom[mask]
        cosv = np.clip(cosv, -1.0, 1.0)
        out[mask] = np.degrees(np.arccos(cosv))
    return out

def geometric_errors_per_sample(
    test_events: pd.DataFrame,
    y_true: pd.DataFrame,
    y_pred: np.ndarray,
) -> pd.DataFrame:
    """Return per-event geometry-aware errors.

    The six raw targets are interpreted as two 3D vectors:
      m = (x_mean, y_mean, z_mean), and h = (x_half_absdiff, y_half_absdiff, z_half_absdiff).
    We also evaluate the displacement vector d = m - mother_xyz, which removes the
    trivial absolute-position component.
    """
    pred = pd.DataFrame(np.asarray(y_pred, dtype=float), columns=EVAL_TARGET_COLS)
    true_mean = y_true[["x_mean", "y_mean", "z_mean"]].to_numpy(dtype=float)
    pred_mean = pred[["x_mean", "y_mean", "z_mean"]].to_numpy(dtype=float)
    mother = test_events[["mother_x", "mother_y", "mother_z"]].to_numpy(dtype=float)
    true_disp = true_mean - mother
    pred_disp = pred_mean - mother
    true_h = y_true[["x_half_absdiff", "y_half_absdiff", "z_half_absdiff"]].to_numpy(dtype=float)
    pred_h = pred[["x_half_absdiff", "y_half_absdiff", "z_half_absdiff"]].to_numpy(dtype=float)
    true_h_norm = _norm_rows(true_h)
    pred_h_norm = _norm_rows(pred_h)
    return pd.DataFrame({
        "mean_position_error": _norm_rows(pred_mean - true_mean),
        "mean_displacement_error": _norm_rows(pred_disp - true_disp),
        "split_vector_error": _norm_rows(pred_h - true_h),
        "split_size_error": np.abs(pred_h_norm - true_h_norm),
        "split_axis_angle_deg": _angle_deg_rows(pred_h, true_h),
    })

def aggregate_geometric_error_df(err_df: pd.DataFrame) -> Dict[str, float]:
    values: Dict[str, float] = {}
    for key in GEOMETRIC_ERROR_KEYS:
        arr = err_df[key].to_numpy(dtype=float) if key in err_df.columns else np.array([], dtype=float)
        values[key] = float(np.nanmean(arr)) if np.isfinite(arr).any() else float("nan")
        values[f"{key}_median"] = float(np.nanmedian(arr)) if np.isfinite(arr).any() else float("nan")
    return values

def _add_skill_columns(row: Dict[str, object], model_vals: Mapping[str, float], ref_vals_by_model: Mapping[str, Mapping[str, float]]) -> None:
    for ref_model, ref_vals in ref_vals_by_model.items():
        for key in GEOMETRIC_ERROR_KEYS:
            row[f"skill_vs_{ref_model}_{key}"] = 1.0 - _safe_div(float(model_vals.get(key, np.nan)), float(ref_vals.get(key, np.nan)))

In [ ]:
def geometric_metric_rows_for_split(
    model_predictions: Mapping[str, np.ndarray],
    test_events: pd.DataFrame,
    y_true: pd.DataFrame,
    dataset_label: str,
    train_size: int,
    repeat: int,
    n_train: int,
    n_test: int,
    split_meta: Optional[Mapping[str, object]] = None,
) -> List[Dict[str, object]]:
    """Layer 2: global geometry-aware metrics for one split."""
    split_meta = dict(split_meta or {})
    vals_by_model: Dict[str, Dict[str, float]] = {}
    for model_name, pred in model_predictions.items():
        vals_by_model[model_name] = aggregate_geometric_error_df(geometric_errors_per_sample(test_events, y_true, pred))
    ref_vals = {m: vals_by_model[m] for m in SKILL_REFERENCE_MODELS if m in vals_by_model}

    rows: List[Dict[str, object]] = []
    for model_name, vals in vals_by_model.items():
        row: Dict[str, object] = {
            "dataset_label": dataset_label,
            "train_size": int(train_size),
            "train_fraction_actual": float(n_train / (n_train + n_test)) if (n_train + n_test) else float("nan"),
            "repeat": repeat,
            "model_name": model_name,
            "metric_layer": "layer2_geometric_global",
            "n_train": int(n_train),
            "n_test": int(n_test),
            "split_unit_actual": split_meta.get("split_unit_actual", ""),
            "split_group_col": split_meta.get("split_group_col", ""),
            "n_train_groups": split_meta.get("n_train_groups", np.nan),
            "n_test_groups": split_meta.get("n_test_groups", np.nan),
        }
        row.update(vals)
        _add_skill_columns(row, vals, ref_vals)
        rows.append(row)
    return rows

In [ ]:
def cellwise_geometric_metric_rows_for_split(
    model_predictions: Mapping[str, np.ndarray],
    test_events: pd.DataFrame,
    y_true: pd.DataFrame,
    dataset_label: str,
    train_size: int,
    repeat: int,
    n_train: int,
    n_test: int,
    split_meta: Optional[Mapping[str, object]] = None,
) -> List[Dict[str, object]]:
    """Layer 3: compute geometry metrics within each mother cell and macro-average across cells.

    This avoids letting the global mixture of different cell types dominate the conclusion.
    The macro row is unweighted across mother cell names present in the test set.
    """
    split_meta = dict(split_meta or {})
    if "mother_name" not in test_events.columns:
        return []

    # Precompute per-sample error tables for every model.
    errors_by_model = {
        model_name: geometric_errors_per_sample(test_events, y_true, pred)
        for model_name, pred in model_predictions.items()
    }

    rows: List[Dict[str, object]] = []
    per_model_cell_vals: Dict[str, List[Dict[str, float]]] = {m: [] for m in model_predictions}
    mother_names = sorted(str(x) for x in pd.Series(test_events["mother_name"]).dropna().unique())

    for mother_name in mother_names:
        idx = test_events.index[test_events["mother_name"].astype(str) == mother_name].to_numpy()
        if len(idx) == 0:
            continue
        vals_by_model: Dict[str, Dict[str, float]] = {}
        for model_name, err_df in errors_by_model.items():
            vals = aggregate_geometric_error_df(err_df.iloc[idx].reset_index(drop=True))
            vals_by_model[model_name] = vals
            per_model_cell_vals[model_name].append(vals)
        ref_vals = {m: vals_by_model[m] for m in SKILL_REFERENCE_MODELS if m in vals_by_model}
        for model_name, vals in vals_by_model.items():
            row: Dict[str, object] = {
                "dataset_label": dataset_label,
                "model_name": model_name,
                "train_size": int(train_size),
                "train_fraction_actual": float(n_train / (n_train + n_test)) if (n_train + n_test) else float("nan"),
                "repeat": repeat,
                "model_name": model_name,
                "metric_layer": "layer3_cellwise",
                "row_type": "mother_cell",
                "mother_name": mother_name,
                "n_cell_test_events": int(len(idx)),
                "n_train": int(n_train),
                "n_test": int(n_test),
                "split_unit_actual": split_meta.get("split_unit_actual", ""),
                "split_group_col": split_meta.get("split_group_col", ""),
                "n_train_groups": split_meta.get("n_train_groups", np.nan),
                "n_test_groups": split_meta.get("n_test_groups", np.nan),
            }
            row.update(vals)
            _add_skill_columns(row, vals, ref_vals)
            rows.append(row)

    # Unweighted macro over mother cells.  Skill is computed from macro errors,
    # not by averaging individual skill scores, so the interpretation remains:
    # 1 - macro_error_model / macro_error_reference.
    macro_vals_by_model: Dict[str, Dict[str, float]] = {}
    for model_name, vals_list in per_model_cell_vals.items():
        if not vals_list:
            continue
        macro_vals: Dict[str, float] = {}
        for key in GEOMETRIC_ERROR_KEYS + GEOMETRIC_EXTRA_KEYS:
            arr = np.array([v.get(key, np.nan) for v in vals_list], dtype=float)
            macro_vals[key] = float(np.nanmean(arr)) if np.isfinite(arr).any() else float("nan")
        macro_vals_by_model[model_name] = macro_vals
    ref_macro_vals = {m: macro_vals_by_model[m] for m in SKILL_REFERENCE_MODELS if m in macro_vals_by_model}
    for model_name, vals in macro_vals_by_model.items():
        row = {
            "dataset_label": dataset_label,
            "train_size": int(train_size),
            "train_fraction_actual": float(n_train / (n_train + n_test)) if (n_train + n_test) else float("nan"),
            "repeat": repeat,
            "model_name": model_name,
            "metric_layer": "layer3_cellwise",
            "row_type": "cell_macro",
            "mother_name": "cell_macro",
            "n_cell_test_events": int(len(mother_names)),
            "n_train": int(n_train),
            "n_test": int(n_test),
            "split_unit_actual": split_meta.get("split_unit_actual", ""),
            "split_group_col": split_meta.get("split_group_col", ""),
            "n_train_groups": split_meta.get("n_train_groups", np.nan),
            "n_test_groups": split_meta.get("n_test_groups", np.nan),
        }
        row.update(vals)
        _add_skill_columns(row, vals, ref_macro_vals)
        rows.append(row)
    return rows

In [ ]:
def summarize_layer_metrics(df: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    non_metric_cols = set(group_cols) | {
        "repeat", "metric_layer", "row_type", "train_fraction_actual", "n_train", "n_test",
        "split_unit_actual", "split_group_col", "n_train_groups", "n_test_groups", "n_cell_test_events",
    }
    metric_cols = [
        c for c in df.columns
        if c not in non_metric_cols and pd.api.types.is_numeric_dtype(df[c])
    ]
    agg_spec = dict(
        n_repeats=("repeat", "nunique"),
        train_fraction_actual_mean=("train_fraction_actual", "mean"),
        n_train_mean=("n_train", "mean"),
        n_test_mean=("n_test", "mean"),
    )
    if "n_cell_test_events" in df.columns:
        agg_spec["n_cell_test_events_mean"] = ("n_cell_test_events", "mean")
    for c in metric_cols:
        agg_spec[f"{c}_mean"] = (c, "mean")
        agg_spec[f"{c}_std"] = (c, "std")
    return df.groupby(list(group_cols), dropna=False).agg(**agg_spec).reset_index()

## 7. Mother-Coordinate Residualization, Prediction Reconstruction, and Train/Test Splits

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def fit_mother_axis_coefficients(
    train_events: pd.DataFrame,
    feature_columns_available: Optional[Sequence[str]] = None,
) -> Tuple[Dict[str, float], List[Dict[str, object]]]:
    """Fit y_mean ≈ c * mother_axis on the training split only.

    The fit deliberately uses one scalar coefficient c and no intercept, matching
    the requested logic.  If a mother_* column is unavailable, c is set to 0 and
    the corresponding delta target remains the raw mean.  In the normal raw
    CellData pipeline mother_x/mother_y/mother_z are always available.
    """
    available = set(feature_columns_available or train_events.columns)
    coeffs: Dict[str, float] = {}
    rows: List[Dict[str, object]] = []
    for spec in MEAN_RESIDUAL_SPECS:
        axis = spec["axis"]
        mean_col = spec["mean_col"]
        mother_col = spec["mother_col"]
        has_mother_feature = mother_col in available and mother_col in train_events.columns
        if has_mother_feature:
            x = train_events[mother_col].to_numpy(dtype=float)
            y = train_events[mean_col].to_numpy(dtype=float)
            denom = float(np.dot(x, x))
            c = float(np.dot(x, y) / denom) if denom > 1e-14 else 0.0
        else:
            denom = float("nan")
            c = 0.0
        coeffs[axis] = c
        rows.append({
            "axis": axis,
            "mean_col": mean_col,
            "mother_col": mother_col,
            "delta_col": spec["delta_col"],
            "has_mother_feature": bool(has_mother_feature),
            "mother_axis_coefficient_c": c,
            "fit_type": "no_intercept_least_squares_y_mean_on_mother_axis",
            "denominator_sum_mother_axis_sq": denom,
            "n_train_for_c": int(len(train_events)),
        })
    return coeffs, rows

def apply_mother_axis_residual_targets(
    events: pd.DataFrame,
    coeffs: Mapping[str, float],
) -> pd.DataFrame:
    """Return a copy whose delta_*_mean columns use y_mean - c*mother_*.

    The half_absdiff target columns are unchanged.
    """
    out = events.copy()
    for spec in MEAN_RESIDUAL_SPECS:
        axis = spec["axis"]
        c = float(coeffs.get(axis, 0.0))
        out[spec["delta_col"]] = out[spec["mean_col"]].to_numpy(dtype=float) - c * out[spec["mother_col"]].to_numpy(dtype=float)
    return out

In [ ]:
def reconstruct_eval_predictions_from_delta(
    test_events: pd.DataFrame,
    pred_model_targets: np.ndarray,
    coeffs: Mapping[str, float],
) -> np.ndarray:
    """Convert model predictions on TARGET_COLS to EVAL_TARGET_COLS.

    The three mean predictions are reconstructed by adding c*mother_axis back.
    Half-difference predictions are passed through unchanged.
    """
    pred_model_targets = np.asarray(pred_model_targets, dtype=float)
    if pred_model_targets.ndim != 2 or pred_model_targets.shape[1] != len(TARGET_COLS):
        raise ValueError(
            f"pred_model_targets must have shape (n, {len(TARGET_COLS)}), got {pred_model_targets.shape}"
        )
    out = np.zeros((len(test_events), len(EVAL_TARGET_COLS)), dtype=float)
    # x mean, y mean, z mean after reconstruction.
    for spec in MEAN_RESIDUAL_SPECS:
        axis = spec["axis"]
        mean_eval_index = EVAL_TARGET_COLS.index(spec["mean_col"])
        model_index = int(spec["target_index"])
        c = float(coeffs.get(axis, 0.0))
        out[:, mean_eval_index] = pred_model_targets[:, model_index] + c * test_events[spec["mother_col"]].to_numpy(dtype=float)
    # Half-difference targets retain the same positions as in TARGET_COLS/EVAL_TARGET_COLS.
    out[:, EVAL_TARGET_COLS.index("x_half_absdiff")] = pred_model_targets[:, TARGET_COLS.index("x_half_absdiff")]
    out[:, EVAL_TARGET_COLS.index("y_half_absdiff")] = pred_model_targets[:, TARGET_COLS.index("y_half_absdiff")]
    out[:, EVAL_TARGET_COLS.index("z_half_absdiff")] = pred_model_targets[:, TARGET_COLS.index("z_half_absdiff")]
    return out

def compute_event_level_train_test_indices(n: int, train_frac: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    """Event-level split. Kept as an explicit option, but not the default.

    In this project, one CellData_*.csv file usually contributes multiple
    division events. A pure event-level split can put different events from the
    same embryo/file into both train and test. That is often too optimistic.
    """
    if n < 2:
        raise ValueError("At least 2 event samples are required for a train/test split")
    n_train = int(round(n * float(train_frac)))
    n_train = max(1, min(n_train, n - 1))
    all_idx = np.arange(n)
    train_idx, test_idx = train_test_split(
        all_idx,
        train_size=n_train,
        random_state=seed,
        shuffle=True,
    )
    return np.asarray(train_idx, dtype=int), np.asarray(test_idx, dtype=int)

def _ordered_unique_groups(events: pd.DataFrame, group_col: str) -> List[str]:
    """Return groups in numeric sample order whenever group_idx is available.

    A plain random split over CellData_0.csv ... CellData_217.csv can still be
    optimistic if neighbouring files are near-duplicates or time-adjacent
    frames.  Therefore v7 supports contiguous/block splits and needs a stable
    biological/sample order.
    """
    if group_col not in events.columns:
        raise KeyError(f"group split column does not exist: {group_col}")
    tmp = events[[group_col]].copy()
    tmp[group_col] = tmp[group_col].astype(str)
    if "group_idx" in events.columns:
        tmp["_group_order"] = pd.to_numeric(events["group_idx"], errors="coerce")
    else:
        tmp["_group_order"] = np.nan
    tmp = tmp.groupby(group_col, as_index=False)["_group_order"].min()
    if tmp["_group_order"].notna().any():
        tmp = tmp.sort_values(["_group_order", group_col], kind="mergesort")
    else:
        tmp = tmp.sort_values(group_col, kind="mergesort")
    return tmp[group_col].astype(str).tolist()

In [ ]:
def _indices_from_group_sets(
    events: pd.DataFrame,
    group_col: str,
    train_groups: Sequence[object],
    test_groups: Sequence[object],
) -> Tuple[np.ndarray, np.ndarray]:
    group_vals = events[group_col].astype(str)
    train_group_set = set(map(str, train_groups))
    test_group_set = set(map(str, test_groups))
    overlap = train_group_set & test_group_set
    if overlap:
        raise RuntimeError(f"train/test group overlap detected: {sorted(overlap)[:5]}")
    train_idx = events.index[group_vals.isin(train_group_set)].to_numpy(dtype=int)
    test_idx = events.index[group_vals.isin(test_group_set)].to_numpy(dtype=int)
    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError("split produced an empty train or test set; check the data, train_frac, or split parameters.")
    return train_idx, test_idx

def compute_group_random_train_test_indices(
    events: pd.DataFrame,
    train_frac: float,
    seed: int,
    group_col: str = "file_name",
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict[str, object]]:
    """Old v6 behaviour: random split by CellData file/group.

    This prevents same-file event leakage, but can still be optimistic when the
    files are ordered frames/near-duplicates.  Use group_contiguous or
    group_blocked for a stricter test.
    """
    groups = _ordered_unique_groups(events, group_col)
    if len(groups) < 2:
        raise ValueError(f"At least 2 distinct {group_col} are required; currently only {len(groups)}.")
    n_train_groups = int(round(len(groups) * float(train_frac)))
    n_train_groups = max(1, min(n_train_groups, len(groups) - 1))
    train_groups, test_groups = train_test_split(
        np.asarray(groups, dtype=object),
        train_size=n_train_groups,
        random_state=seed,
        shuffle=True,
    )
    train_idx, test_idx = _indices_from_group_sets(events, group_col, train_groups, test_groups)
    meta = {"n_purged_groups": 0, "split_strategy_detail": "random groups"}
    return train_idx, test_idx, list(map(str, train_groups)), list(map(str, test_groups)), meta

def compute_group_contiguous_train_test_indices(
    events: pd.DataFrame,
    train_frac: float,
    seed: int,
    group_col: str = "file_name",
    purge_gap_groups: int = 0,
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict[str, object]]:
    """Train on one contiguous window of groups; test on the remaining groups.

    This is the v7 default.  It is stricter than random group split for data
    files like CellData_0.csv ... CellData_217.csv, where neighbouring indices
    may be highly similar.  Repeats choose different contiguous windows.
    """
    groups = _ordered_unique_groups(events, group_col)
    if len(groups) < 2:
        raise ValueError(f"At least 2 distinct {group_col} are required; currently only {len(groups)}.")
    n_train_groups = int(round(len(groups) * float(train_frac)))
    n_train_groups = max(1, min(n_train_groups, len(groups) - 1))
    rng = np.random.default_rng(seed)
    max_start = len(groups) - n_train_groups
    start = int(rng.integers(0, max_start + 1)) if max_start > 0 else 0
    end = start + n_train_groups
    train_groups = groups[start:end]
    purge_start = max(0, start - int(purge_gap_groups))
    purge_end = min(len(groups), end + int(purge_gap_groups))
    purged = set(groups[purge_start:start] + groups[end:purge_end])
    train_set = set(train_groups)
    test_groups = [g for g in groups if g not in train_set and g not in purged]
    train_idx, test_idx = _indices_from_group_sets(events, group_col, train_groups, test_groups)
    meta = {
        "n_purged_groups": len(purged),
        "split_strategy_detail": f"contiguous train window [{start}, {end}), purge_gap={purge_gap_groups}",
    }
    return train_idx, test_idx, train_groups, test_groups, meta

In [ ]:
def compute_group_prefix_train_test_indices(
    events: pd.DataFrame,
    train_frac: float,
    group_col: str = "file_name",
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict[str, object]]:
    """Chronological/prefix split: first fraction of groups for train, rest for test."""
    groups = _ordered_unique_groups(events, group_col)
    if len(groups) < 2:
        raise ValueError(f"At least 2 distinct {group_col} are required; currently only {len(groups)}.")
    n_train_groups = int(round(len(groups) * float(train_frac)))
    n_train_groups = max(1, min(n_train_groups, len(groups) - 1))
    train_groups = groups[:n_train_groups]
    test_groups = groups[n_train_groups:]
    train_idx, test_idx = _indices_from_group_sets(events, group_col, train_groups, test_groups)
    meta = {"n_purged_groups": 0, "split_strategy_detail": "prefix/chronological split"}
    return train_idx, test_idx, train_groups, test_groups, meta

def compute_group_blocked_train_test_indices(
    events: pd.DataFrame,
    train_frac: float,
    seed: int,
    group_col: str = "file_name",
    block_size: int = 10,
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict[str, object]]:
    """Split contiguous blocks of groups, not individual groups."""
    groups = _ordered_unique_groups(events, group_col)
    if len(groups) < 2:
        raise ValueError(f"At least 2 distinct {group_col} are required; currently only {len(groups)}.")
    block_size = max(1, int(block_size))
    blocks = [groups[i:i + block_size] for i in range(0, len(groups), block_size)]
    if len(blocks) < 2:
        raise ValueError("block_size is too large and leaves only one block; reduce --block-size.")
    n_train_blocks = int(round(len(blocks) * float(train_frac)))
    n_train_blocks = max(1, min(n_train_blocks, len(blocks) - 1))
    rng = np.random.default_rng(seed)
    train_block_idx = set(rng.choice(np.arange(len(blocks)), size=n_train_blocks, replace=False).tolist())
    train_groups = [g for i, b in enumerate(blocks) if i in train_block_idx for g in b]
    test_groups = [g for i, b in enumerate(blocks) if i not in train_block_idx for g in b]
    train_idx, test_idx = _indices_from_group_sets(events, group_col, train_groups, test_groups)
    meta = {
        "n_purged_groups": 0,
        "split_strategy_detail": f"random contiguous blocks, block_size={block_size}, n_train_blocks={n_train_blocks}/{len(blocks)}",
    }
    return train_idx, test_idx, train_groups, test_groups, meta

In [ ]:
def compute_train_test_split(
    events: pd.DataFrame,
    train_frac: float,
    seed: int,
    split_unit: str = "group_contiguous",
    group_col: str = "file_name",
    block_size: int = 10,
    purge_gap_groups: int = 0,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, object]]:
    """Return train/test row indices plus split metadata.

    split_unit options:
      - event: row-level random split; useful only for single-file debugging.
      - group_random or group: old v6 file-level random split.
      - group_contiguous: default; one contiguous window of CellData indices.
      - group_prefix: first train_frac of ordered files as train, rest as test.
      - group_blocked: random split of contiguous file blocks.
    """
    if split_unit == "event":
        train_idx, test_idx = compute_event_level_train_test_indices(len(events), train_frac, seed)
        return train_idx, test_idx, {
            "split_unit_actual": "event",
            "split_group_col": "",
            "n_train_groups": np.nan,
            "n_test_groups": np.nan,
            "n_purged_groups": 0,
            "train_groups": "",
            "test_groups": "",
            "split_strategy_detail": "event random split",
        }

    if split_unit in {"group", "group_random"}:
        train_idx, test_idx, train_groups, test_groups, extra = compute_group_random_train_test_indices(
            events, train_frac=train_frac, seed=seed, group_col=group_col
        )
        actual = "group_random"
    elif split_unit == "group_contiguous":
        train_idx, test_idx, train_groups, test_groups, extra = compute_group_contiguous_train_test_indices(
            events, train_frac=train_frac, seed=seed, group_col=group_col, purge_gap_groups=purge_gap_groups
        )
        actual = "group_contiguous"
    elif split_unit == "group_prefix":
        train_idx, test_idx, train_groups, test_groups, extra = compute_group_prefix_train_test_indices(
            events, train_frac=train_frac, group_col=group_col
        )
        actual = "group_prefix"
    elif split_unit == "group_blocked":
        train_idx, test_idx, train_groups, test_groups, extra = compute_group_blocked_train_test_indices(
            events, train_frac=train_frac, seed=seed, group_col=group_col, block_size=block_size
        )
        actual = "group_blocked"
    else:
        raise ValueError(
            f"Unknown split_unit: {split_unit}; options event, group_random, group, group_contiguous, group_prefix, group_blocked"
        )

    return train_idx, test_idx, {
        "split_unit_actual": actual,
        "split_group_col": group_col,
        "n_train_groups": len(train_groups),
        "n_test_groups": len(test_groups),
        "n_purged_groups": extra.get("n_purged_groups", 0),
        "train_groups": ";".join(map(str, train_groups)),
        "test_groups": ";".join(map(str, test_groups)),
        "split_strategy_detail": extra.get("split_strategy_detail", ""),
    }

## 8. FixedTerm Term Selection, Equation Parsing, and Per-Target OLS Fitting

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def _nonconstant_columns(X: pd.DataFrame, tol: float = 1e-14) -> List[str]:
    cols: List[str] = []
    for c in X.columns:
        v = X[c].to_numpy(dtype=float)
        if np.nanstd(v) > tol and np.all(np.isfinite(v)):
            cols.append(c)
    return cols

In [ ]:
def select_terms_from_source_lasso_or_correlation(
    source_events: pd.DataFrame,
    source_feature_df: pd.DataFrame,
    max_terms_per_target: int,
    random_state: int,
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    """Fallback term selection from source feature library if no equation dir is supplied."""
    feature_cols = feature_value_columns(source_feature_df)
    X_terms = source_feature_df[feature_cols].copy()
    y_df = source_events[TARGET_COLS].copy()
    available_cols = _nonconstant_columns(X_terms)
    if not available_cols:
        raise ValueError("source feature library has no usable non-constant columns")

    selected: Dict[str, List[str]] = {}
    report_rows: List[Dict[str, object]] = []
    for target in TARGET_COLS:
        y = y_df[target].to_numpy(dtype=float)
        X_avail = X_terms[available_cols]
        n = len(y)
        chosen: List[str] = []
        method = "correlation_fallback"

        if n >= 8 and len(available_cols) >= 2:
            cv = min(5, n)
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=ConvergenceWarning)
                    model = make_pipeline(
                        StandardScaler(),
                        LassoCV(
                            cv=cv,
                            random_state=random_state,
                            max_iter=20000,
                            n_jobs=None,
                            fit_intercept=True,
                        ),
                    )
                    model.fit(X_avail.to_numpy(dtype=float), y)
                lasso = model.named_steps["lassocv"]
                coefs = np.asarray(lasso.coef_, dtype=float)
                nz = np.where(np.abs(coefs) > 1e-10)[0]
                if len(nz) > 0:
                    ranked_idx = nz[np.argsort(np.abs(coefs[nz]))[::-1]]
                    chosen = [available_cols[i] for i in ranked_idx[:max_terms_per_target]]
                    method = "LassoCV_nonzero"
            except Exception:
                chosen = []

        if not chosen:
            scores = []
            for c in available_cols:
                xv = X_avail[c].to_numpy(dtype=float)
                if np.nanstd(xv) <= 1e-14 or np.nanstd(y) <= 1e-14:
                    score = 0.0
                else:
                    score = abs(float(np.corrcoef(xv, y)[0, 1]))
                    if not np.isfinite(score):
                        score = 0.0
                scores.append((score, c))
            scores.sort(key=lambda z: (z[0], z[1]), reverse=True)
            chosen = [c for _, c in scores[:max_terms_per_target]]
            method = "abs_correlation_topk_on_CellFeatureExtractor_library"

        selected[target] = chosen
        for order, feat in enumerate(chosen, start=1):
            report_rows.append({
                "target": target,
                "target_display": TARGET_DISPLAY[target],
                "order": order,
                "feature_name": feat,
                "selection_method": method,
                "source_n_events": n,
                "max_terms_per_target": max_terms_per_target,
            })

    return selected, pd.DataFrame(report_rows)

def _normalize_equation_lhs(lhs: str) -> str:
    return re.sub(r"\s+", "", str(lhs).replace("−", "-").replace("×", "*"))

In [ ]:
def infer_target_from_equation_lhs(lhs: str, order_idx: int) -> str:
    norm = _normalize_equation_lhs(lhs)
    if "(x1+x2)/2" in norm or "x_mean" in norm:
        return "x_mean"
    if "|x1-x2|/2" in norm or "x_half" in norm:
        return "x_half_absdiff"
    if "(y1+y2)/2" in norm or "y_mean" in norm:
        return "y_mean"
    if "|y1-y2|/2" in norm or "y_half" in norm:
        return "y_half_absdiff"
    if "(z1+z2)/2" in norm or "z_mean" in norm:
        return "z_mean"
    if "|z1-z2|/2" in norm or "z_half" in norm:
        return "z_half_absdiff"
    if 0 <= order_idx < len(EVAL_TARGET_COLS):
        return EVAL_TARGET_COLS[order_idx]
    raise ValueError(f"Unable to identify target from equation left-hand side: {lhs}")

def read_regression_equation_strings(path: str) -> List[str]:
    import csv
    equations: List[str] = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        for row in csv.reader(f):
            for cell in row:
                text = str(cell).strip()
                if "=" in text:
                    equations.append(text)
    if not equations:
        raise ValueError(f"{path} does not contain an equation of the form lhs = rhs")
    return equations

def extract_term_names_from_rhs(rhs: str) -> List[str]:
    """Extract feature names from RHS and ignore numeric coefficients.

    Examples:
        0.867×x - 0.008×x_adj_xj_pow2 -> [x, x_adj_xj_pow2]
    """
    rhs = str(rhs).replace("−", "-").replace("×", "*")
    # Normalize subtraction as + negative term, then split additive pieces.
    pieces = re.split(r"(?=[+-])", rhs)
    terms: List[str] = []
    for piece in pieces:
        seg = piece.strip()
        if not seg:
            continue
        seg = re.sub(r"^[+]\s*", "", seg)
        seg = re.sub(r"^-\s*", "", seg)
        # Remove a leading coefficient and multiplication symbol if present.
        seg = re.sub(
            r"^(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?\s*(?:\*)\s*",
            "",
            seg,
        ).strip()
        # If there is still a coefficient without an explicit *, remove it only
        # when followed by whitespace and a variable name.
        seg = re.sub(
            r"^(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?\s+",
            "",
            seg,
        ).strip()
        m = re.search(r"[A-Za-z0-9_]*[A-Za-z_][A-Za-z0-9_]*", seg)
        if m:
            term = m.group(0)
            if term not in terms:
                terms.append(term)
    return terms

In [ ]:
def load_terms_from_regression_equations(
    regression_equations_csv: str,
    term_library_columns: Sequence[str],
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    """Load selected term names from regression_equations.csv.

    Coefficients in the CSV are deliberately ignored.  The returned terms are
    canonicalized against the current CellFeatureExtractor feature library.
    """
    path = Path(regression_equations_csv)
    if not path.exists():
        raise FileNotFoundError(f"regression_equations.csv does not exist: {regression_equations_csv}")
    equations = read_regression_equation_strings(str(path))
    selected: Dict[str, List[str]] = {t: [] for t in EVAL_TARGET_COLS}
    report_rows: List[Dict[str, object]] = []
    dropped_rows: List[Dict[str, object]] = []
    seen_targets: set = set()

    for order_idx, eq in enumerate(equations):
        lhs, rhs = eq.split("=", 1)
        target = infer_target_from_equation_lhs(lhs, order_idx)
        raw_terms = extract_term_names_from_rhs(rhs)
        if not raw_terms:
            raise ValueError(f"No terms could be parsed from equation: {eq}")
        seen_targets.add(target)
        keep: List[str] = []
        for raw in raw_terms:
            matched = canonicalize_feature_name(raw, term_library_columns)
            if matched is not None:
                if matched not in keep:
                    keep.append(matched)
            else:
                dropped_rows.append({
                    "target": target,
                    "target_display": TARGET_DISPLAY[target],
                    "source_equation": eq,
                    "original_feature_name": raw,
                    "reason": "not present in CellFeatureExtractor feature library under current toggles",
                })
        if not keep:
            raise ValueError(
                f"{target} regression_equations equation has no computable terms; original equation: {eq}."
                "Check feature toggles, adjacency matrices, or term names."
            )
        selected[target] = keep
        for i, feat in enumerate(keep, start=1):
            report_rows.append({
                "target": target,
                "target_display": TARGET_DISPLAY[target],
                "order": i,
                "feature_name": feat,
                "original_terms_in_equation": ",".join(raw_terms),
                "source_equation": eq,
                "source_file": str(path),
                "selection_method": "loaded_from_regression_equations_csv_coefficients_ignored",
                "coefficient_from_csv_used": False,
            })

    missing_targets = [t for t in EVAL_TARGET_COLS if t not in seen_targets]
    if missing_targets:
        raise ValueError(
            f"regression_equations.csv does not cover these targets: {missing_targets}."
            "Six equations are required: x_mean/x_half_absdiff/y_mean/y_half_absdiff/z_mean/z_half_absdiff."
        )
    report = pd.DataFrame(report_rows)
    if dropped_rows:
        report.attrs["dropped_terms"] = pd.DataFrame(dropped_rows)
    return selected, report

def _equation_task_path(term_equation_dir: Path, task_idx: int) -> Optional[Path]:
    candidates = [
        term_equation_dir / "equations" / f"task_{task_idx}_equation_coefficients.csv",
        term_equation_dir / f"task_{task_idx}_equation_coefficients.csv",
        term_equation_dir / f"task_{task_idx}.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

In [ ]:
def canonicalize_feature_name(feature_name: str, available: Sequence[str]) -> Optional[str]:
    available_set = set(available)
    feat = str(feature_name).strip()
    if not feat or feat.lower() in {"nan", "none", "__intercept__", "intercept", "const", "constant"}:
        return None
    candidates = [feat]
    if not feat.startswith("mother_"):
        candidates.append("mother_" + feat)
    if feat.startswith("mother_"):
        candidates.append(feat.replace("mother_", "", 1))
    # A few older notebooks used upper-case coordinate names.
    candidates.extend([c.replace("mother_X", "mother_x").replace("mother_Y", "mother_y").replace("mother_Z", "mother_z") for c in list(candidates)])
    for c in candidates:
        if c in available_set:
            return c
    return None

def load_terms_from_equation_dir(
    term_equation_dir: str,
    term_library_columns: Sequence[str],
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    base = Path(term_equation_dir)
    if not base.exists():
        raise FileNotFoundError(f"term equation dir does not exist: {term_equation_dir}")

    selected: Dict[str, List[str]] = {}
    report_rows: List[Dict[str, object]] = []
    dropped_rows: List[Dict[str, object]] = []

    for task_idx, target in enumerate(TARGET_COLS, start=1):
        p = _equation_task_path(base, task_idx)
        if p is None:
            raise FileNotFoundError(f"Could not find task_{task_idx}_equation_coefficients.csv in {term_equation_dir}")
        df = pd.read_csv(p)
        if "feature_name" in df.columns:
            feats = df["feature_name"].astype(str).tolist()
        elif "feature" in df.columns:
            feats = df["feature"].astype(str).tolist()
        elif "term" in df.columns:
            feats = df["term"].astype(str).tolist()
        else:
            feats = df.iloc[:, 0].astype(str).tolist()

        keep: List[str] = []
        for feat in feats:
            matched = canonicalize_feature_name(feat, term_library_columns)
            if matched is not None:
                if matched not in keep:
                    keep.append(matched)
            else:
                dropped_rows.append({
                    "target": target,
                    "target_display": TARGET_DISPLAY[target],
                    "original_feature_name": str(feat),
                    "reason": "not present in CellFeatureExtractor feature library under current toggles",
                })
        if not keep:
            raise ValueError(
                f"task {task_idx} ({target}) has no terms from the existing equation that can be computed in the current CellFeatureExtractor feature library;"
                "Check --feature-toggles, adjacency feature names, or use --learn-terms-from-source."
            )
        selected[target] = keep
        for order, feat in enumerate(keep, start=1):
            report_rows.append({
                "target": target,
                "target_display": TARGET_DISPLAY[target],
                "order": order,
                "feature_name": feat,
                "selection_method": "loaded_from_equation_dir_CellFeatureExtractor_library",
                "source_file": str(p),
            })

    report = pd.DataFrame(report_rows)
    if dropped_rows:
        report.attrs["dropped_terms"] = pd.DataFrame(dropped_rows)
    return selected, report

In [ ]:
def resolve_terms_against_eval_features(
    selected_terms: Mapping[str, List[str]],
    eval_feature_columns: Sequence[str],
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    resolved: Dict[str, List[str]] = {}
    rows: List[Dict[str, object]] = []
    for target, feats in selected_terms.items():
        current: List[str] = []
        for feat in feats:
            matched = canonicalize_feature_name(feat, eval_feature_columns)
            rows.append({
                "target": target,
                "target_display": TARGET_DISPLAY[target],
                "feature_name_from_source": feat,
                "feature_name_in_eval": matched if matched is not None else "",
                "present_in_eval": matched is not None,
                "is_adjacent_feature": "_adj_" in str(feat),
            })
            if matched is not None and matched not in current:
                current.append(matched)
        if not current:
            raise KeyError(f"target={target} has no usable fixed terms in the eval feature library")
        resolved[target] = current
    return resolved, pd.DataFrame(rows)

In [ ]:
def fit_fixed_term_ols_predict(
    selected_terms: Mapping[str, List[str]],
    train_events: pd.DataFrame,
    test_events: pd.DataFrame,
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    fit_intercept: bool,
    dataset_label: str,
    train_size: int,
    repeat: int,
    split_meta: Optional[Mapping[str, object]] = None,
    model_name: str = "FixedTerm_OLS",
) -> Tuple[np.ndarray, List[Dict[str, object]]]:
    split_meta = dict(split_meta or {})
    preds = []
    coef_rows: List[Dict[str, object]] = []
    for target in TARGET_COLS:
        feats = selected_terms[target]
        missing = [f for f in feats if f not in train_features.columns]
        if missing:
            raise KeyError(f"FixedTerm is missing term columns for {target}: {missing}")
        X_train = train_features[feats].to_numpy(dtype=float)
        y_train = train_events[target].to_numpy(dtype=float)
        X_test = test_features[feats].to_numpy(dtype=float)
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
        X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
        reg = LinearRegression(fit_intercept=fit_intercept)
        reg.fit(X_train, y_train)
        train_pred = reg.predict(X_train)
        train_mse = float(mean_squared_error(y_train, train_pred))
        train_r2 = safe_r2(y_train, train_pred)
        preds.append(reg.predict(X_test))
        for order, (feat, coef) in enumerate(zip(feats, reg.coef_), start=1):
            coef_rows.append({
                "dataset_label": dataset_label,
                "model_name": model_name,
                "train_size": int(train_size),
                "repeat": repeat,
                "target": target,
                "target_display": TARGET_DISPLAY[target],
                "order": order,
                "feature_name": feat,
                "coefficient": float(coef),
                "intercept": float(reg.intercept_) if fit_intercept else 0.0,
                "n_train": len(train_events),
                "n_test": len(test_events),
                "split_unit_actual": split_meta.get("split_unit_actual", ""),
                "split_group_col": split_meta.get("split_group_col", ""),
                "n_train_groups": split_meta.get("n_train_groups", np.nan),
                "n_test_groups": split_meta.get("n_test_groups", np.nan),
                "is_adjacent_feature": "_adj_" in feat,
                "is_same_time_context_feature": str(feat).startswith("ftctx_"),
                "target_fit_is_independent": True,
                "least_squares_objective": "minimize training MSE for this single target column over the selected fixed terms",
                "train_mse_for_target": train_mse,
                "train_r2_for_target": train_r2,
            })
    return np.column_stack(preds), coef_rows

## 9. Baseline Models and Random Train-Size Splits

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def make_residual_mean_target_matrix(events: pd.DataFrame) -> np.ndarray:
    """Targets for residual-mean baseline variants.

    For mean-position coordinates, fit x_mean-mother_x, y_mean-mother_y,
    z_mean-mother_z.  For half-difference coordinates, fit the raw targets.
    """
    return np.column_stack([
        events["x_mean"].to_numpy(dtype=float) - events["mother_x"].to_numpy(dtype=float),
        events["x_half_absdiff"].to_numpy(dtype=float),
        events["y_mean"].to_numpy(dtype=float) - events["mother_y"].to_numpy(dtype=float),
        events["y_half_absdiff"].to_numpy(dtype=float),
        events["z_mean"].to_numpy(dtype=float) - events["mother_z"].to_numpy(dtype=float),
        events["z_half_absdiff"].to_numpy(dtype=float),
    ])

def make_raw_target_matrix(events: pd.DataFrame) -> np.ndarray:
    """Raw six-dimensional targets: x_mean/x_half/y_mean/y_half/z_mean/z_half."""
    return events[EVAL_TARGET_COLS].to_numpy(dtype=float)

def reconstruct_residual_mean_predictions(test_events: pd.DataFrame, pred_model_targets: np.ndarray) -> np.ndarray:
    pred_model_targets = np.asarray(pred_model_targets, dtype=float)
    if pred_model_targets.ndim != 2 or pred_model_targets.shape[1] != len(BASELINE_MODEL_TARGET_COLS):
        raise ValueError(
            f"residual-mean pred_model_targets must have shape (n, {len(BASELINE_MODEL_TARGET_COLS)}), "
            f"got {pred_model_targets.shape}"
        )
    out = np.zeros((len(test_events), len(EVAL_TARGET_COLS)), dtype=float)
    out[:, EVAL_TARGET_COLS.index("x_mean")] = pred_model_targets[:, 0] + test_events["mother_x"].to_numpy(dtype=float)
    out[:, EVAL_TARGET_COLS.index("x_half_absdiff")] = pred_model_targets[:, 1]
    out[:, EVAL_TARGET_COLS.index("y_mean")] = pred_model_targets[:, 2] + test_events["mother_y"].to_numpy(dtype=float)
    out[:, EVAL_TARGET_COLS.index("y_half_absdiff")] = pred_model_targets[:, 3]
    out[:, EVAL_TARGET_COLS.index("z_mean")] = pred_model_targets[:, 4] + test_events["mother_z"].to_numpy(dtype=float)
    out[:, EVAL_TARGET_COLS.index("z_half_absdiff")] = pred_model_targets[:, 5]
    return out

def fit_train_mean_baseline_predict(train_events: pd.DataFrame, test_events: pd.DataFrame) -> np.ndarray:
    """Predict the training-set mean of each raw target for every test event."""
    mean_values = train_events[EVAL_TARGET_COLS].mean(axis=0).to_numpy(dtype=float)
    return np.tile(mean_values.reshape(1, -1), (len(test_events), 1))

def fit_mother_copy_baseline_predict(train_events: pd.DataFrame, test_events: pd.DataFrame) -> np.ndarray:
    """Copy mother coordinates for mean targets; use train means for half-diff targets."""
    out = np.zeros((len(test_events), len(EVAL_TARGET_COLS)), dtype=float)
    out[:, EVAL_TARGET_COLS.index("x_mean")] = test_events["mother_x"].to_numpy(dtype=float)
    out[:, EVAL_TARGET_COLS.index("y_mean")] = test_events["mother_y"].to_numpy(dtype=float)
    out[:, EVAL_TARGET_COLS.index("z_mean")] = test_events["mother_z"].to_numpy(dtype=float)
    for col in ["x_half_absdiff", "y_half_absdiff", "z_half_absdiff"]:
        out[:, EVAL_TARGET_COLS.index(col)] = float(train_events[col].mean())
    return out

In [ ]:
def _make_xyz_regressor(model_family: str, random_state: int, mlp_max_iter: int = 120, n_train: Optional[int] = None):
    """Create a same-time-cell-context baseline regressor.

    RandomForest uses a conservative adaptive configuration for this small-data
    setting: more trees for stability, shallow-ish trees when n is tiny, and a
    positive min_samples_leaf so RF cannot memorize every single event too easily.
    """
    if model_family == "RandomForest":
        n = int(n_train or 0)
        leaf = max(1, min(5, int(math.ceil(max(n, 1) * 0.08))))
        max_depth = None if n >= 80 else max(2, min(6, int(math.ceil(math.log2(max(n, 2))))))
        return RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            min_samples_leaf=leaf,
            min_samples_split=max(2, 2 * leaf),
            max_depth=max_depth,
            max_features=0.7,
            bootstrap=True,
            n_jobs=-1,
        )
    if model_family == "MLP":
        return make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(256,),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=int(mlp_max_iter),
                early_stopping=False,
                random_state=random_state,
            ),
        )
    if model_family == "LinearRegression":
        return make_pipeline(StandardScaler(), LinearRegression())
    raise ValueError(f"Unknown baseline model family: {model_family}")

In [ ]:
def fit_xyz_baseline_predict(
    model_name: str,
    train_events: pd.DataFrame,
    test_events: pd.DataFrame,
    random_state: int,
    mlp_max_iter: int = 120,
    train_context_features: Optional[pd.DataFrame] = None,
    test_context_features: Optional[pd.DataFrame] = None,
) -> np.ndarray:
    """Fit one baseline and return raw six-dimensional predictions.

    Supported model names:
      - MotherCopyBaseline: copies mother coordinates for mean targets; halfdiffs use train means.
      - LinearRegression/RandomForest/MLP: direct raw six-target regression from the full
        same-time cell context, not merely mother_x/mother_y/mother_z.

    The six target columns remain independent output coordinates in the sense of
    evaluation.  sklearn's multi-output estimators optimize the sum of per-column
    squared losses; for ordinary least squares this is algebraically equivalent
    to fitting six independent least-squares problems with a shared design matrix.
    """
    missing_targets = [c for c in EVAL_TARGET_COLS if c not in train_events.columns or c not in test_events.columns]
    if missing_targets:
        raise KeyError(f"baseline events are missing target columns: {sorted(set(missing_targets))}")

    if model_name == "MotherCopyBaseline":
        missing_mother = [c for c in XYZ_COLS if c not in train_events.columns or c not in test_events.columns]
        if missing_mother:
            raise KeyError(f"MotherCopyBaseline is missing mother coordinate columns: {sorted(set(missing_mother))}")
        return fit_mother_copy_baseline_predict(train_events, test_events)

    if train_context_features is None:
        train_context_features = build_same_time_cell_context_features(train_events)
    if test_context_features is None:
        # Build from combined events to keep the same context-column universe.
        combined = pd.concat([train_events, test_events], axis=0, ignore_index=True)
        combined_context = build_same_time_cell_context_features(combined)
        train_context_features = combined_context.iloc[:len(train_events)].reset_index(drop=True)
        test_context_features = combined_context.iloc[len(train_events):].reset_index(drop=True)

    context_cols = baseline_context_value_columns(train_context_features)
    if not context_cols:
        raise ValueError("baseline context features are empty; cannot train LinearRegression/RandomForest/MLP.")
    missing_in_test = [c for c in context_cols if c not in test_context_features.columns]
    if missing_in_test:
        raise KeyError(f"test baseline context is missing columns: {missing_in_test[:10]}")

    X_train = train_context_features[context_cols].to_numpy(dtype=float)
    X_test = test_context_features[context_cols].to_numpy(dtype=float)
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

    if model_name == "LinearRegression":
        model_family = "LinearRegression"
    elif model_name == "RandomForest":
        model_family = "RandomForest"
    elif model_name == "MLP":
        model_family = "MLP"
    else:
        raise ValueError(f"Unknown baseline model_name: {model_name}")

    y_train = make_raw_target_matrix(train_events)
    model = _make_xyz_regressor(model_family, random_state=random_state, mlp_max_iter=mlp_max_iter, n_train=len(train_events))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        model.fit(X_train, y_train)
    pred_model_targets = np.asarray(model.predict(X_test), dtype=float)
    if pred_model_targets.ndim == 1:
        pred_model_targets = pred_model_targets.reshape(-1, 1)
    if pred_model_targets.shape != (len(test_events), len(EVAL_TARGET_COLS)):
        raise ValueError(
            f"raw context-baseline prediction shape mismatch for {model_name}: got {pred_model_targets.shape}, "
            f"expected {(len(test_events), len(EVAL_TARGET_COLS))}"
        )
    return pred_model_targets

In [ ]:
def compute_random_cell_train_test_indices_by_size(n: int, train_size: int, seed: int) -> Tuple[np.ndarray, np.ndarray, Dict[str, object]]:
    if train_size <= 0:
        raise ValueError(f"train_size must be a positive integer; received {train_size}")
    if train_size >= n:
        raise ValueError(f"train_size={train_size} is not smaller than total event count n={n}; cannot keep a test set")
    rng = np.random.default_rng(seed)
    train_idx = np.sort(rng.choice(np.arange(n), size=int(train_size), replace=False))
    mask = np.ones(n, dtype=bool)
    mask[train_idx] = False
    test_idx = np.where(mask)[0]
    meta = {
        "split_unit_actual": "cell_random",
        "split_group_col": "event_index",
        "train_size_requested": int(train_size),
        "train_size_actual": int(len(train_idx)),
        "train_fraction_actual": float(len(train_idx) / n),
        "n_train_groups": int(len(train_idx)),
        "n_test_groups": int(len(test_idx)),
        "n_purged_groups": 0,
        "split_strategy_detail": f"randomly selected {len(train_idx)} mother-cell division events as train; all remaining events as test",
    }
    return train_idx, test_idx, meta

## 10. Split Audits, Result Summaries, and Prediction File Output

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def make_split_assignment_rows_by_size(
    events: pd.DataFrame,
    train_idx: Sequence[int],
    test_idx: Sequence[int],
    dataset_label: str,
    train_size: int,
    repeat: int,
    split_meta: Mapping[str, object],
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for split_name, idxs in [("train", train_idx), ("test", test_idx)]:
        part = events.iloc[list(idxs)]
        for event_index, row in part.iterrows():
            rows.append({
                "dataset_label": dataset_label,
                "train_size": int(train_size),
                "train_fraction_actual": split_meta.get("train_fraction_actual", np.nan),
                "repeat": repeat,
                "split": split_name,
                "event_index": int(event_index),
                "sample_id": row.get("sample_id", ""),
                "file_name": row.get("file_name", ""),
                "group_idx": row.get("group_idx", np.nan),
                "mother_name": row.get("mother_name", ""),
                "transition": row.get("transition", ""),
                "split_unit_actual": split_meta.get("split_unit_actual", ""),
                "split_strategy_detail": split_meta.get("split_strategy_detail", ""),
            })
    return rows

def make_leakage_audit_row_by_size(
    train_events: pd.DataFrame,
    test_events: pd.DataFrame,
    dataset_label: str,
    train_size: int,
    repeat: int,
    split_meta: Mapping[str, object],
) -> Dict[str, object]:
    train_sample_ids = set(train_events["sample_id"].astype(str)) if "sample_id" in train_events else set()
    test_sample_ids = set(test_events["sample_id"].astype(str)) if "sample_id" in test_events else set()
    train_groups = set(train_events["file_name"].astype(str)) if "file_name" in train_events else set()
    test_groups = set(test_events["file_name"].astype(str)) if "file_name" in test_events else set()

    def xyz_tuples(df: pd.DataFrame, decimals: Optional[int] = None) -> set:
        arr = df[XYZ_COLS].to_numpy(dtype=float)
        if decimals is not None:
            arr = np.round(arr, decimals=decimals)
        return set(map(tuple, arr.tolist()))

    return {
        "dataset_label": dataset_label,
        "train_size": int(train_size),
        "train_fraction_actual": split_meta.get("train_fraction_actual", np.nan),
        "repeat": repeat,
        "split_unit_actual": split_meta.get("split_unit_actual", ""),
        "split_strategy_detail": split_meta.get("split_strategy_detail", ""),
        "n_train": len(train_events),
        "n_test": len(test_events),
        "sample_id_overlap_count": len(train_sample_ids & test_sample_ids),
        "file_group_overlap_count": len(train_groups & test_groups),
        "exact_mother_xyz_overlap_count": len(xyz_tuples(train_events) & xyz_tuples(test_events)),
        "rounded6_mother_xyz_overlap_count": len(xyz_tuples(train_events, decimals=6) & xyz_tuples(test_events, decimals=6)),
        "baseline_feature_columns": "same_file_same_T_all_cells ctx_*",
        "baseline_model_target_columns": ",".join(BASELINE_MODEL_TARGET_COLS),
        "evaluation_target_columns": ",".join(EVAL_TARGET_COLS),
    }

In [ ]:
def make_split_assignment_rows(
    events: pd.DataFrame,
    train_idx: Sequence[int],
    test_idx: Sequence[int],
    dataset_label: str,
    train_frac: float,
    repeat: int,
    split_meta: Mapping[str, object],
) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for split_name, idxs in [("train", train_idx), ("test", test_idx)]:
        part = events.iloc[list(idxs)]
        for _, row in part.iterrows():
            rows.append({
                "dataset_label": dataset_label,
                "train_frac": train_frac,
                "repeat": repeat,
                "split": split_name,
                "sample_id": row.get("sample_id", ""),
                "file_name": row.get("file_name", ""),
                "group_idx": row.get("group_idx", np.nan),
                "mother_name": row.get("mother_name", ""),
                "transition": row.get("transition", ""),
                "split_unit_actual": split_meta.get("split_unit_actual", ""),
                "split_strategy_detail": split_meta.get("split_strategy_detail", ""),
            })
    return rows

def make_leakage_audit_row(
    train_events: pd.DataFrame,
    test_events: pd.DataFrame,
    dataset_label: str,
    train_frac: float,
    repeat: int,
    split_meta: Mapping[str, object],
) -> Dict[str, object]:
    train_sample_ids = set(train_events["sample_id"].astype(str)) if "sample_id" in train_events else set()
    test_sample_ids = set(test_events["sample_id"].astype(str)) if "sample_id" in test_events else set()
    train_groups = set(train_events["file_name"].astype(str)) if "file_name" in train_events else set()
    test_groups = set(test_events["file_name"].astype(str)) if "file_name" in test_events else set()

    def xyz_tuples(df: pd.DataFrame, decimals: Optional[int] = None) -> set:
        arr = df[XYZ_COLS].to_numpy(dtype=float)
        if decimals is not None:
            arr = np.round(arr, decimals=decimals)
        return set(map(tuple, arr.tolist()))

    exact_overlap = len(xyz_tuples(train_events) & xyz_tuples(test_events))
    rounded_overlap = len(xyz_tuples(train_events, decimals=6) & xyz_tuples(test_events, decimals=6))

    min_group_idx_gap = np.nan
    if "group_idx" in train_events.columns and "group_idx" in test_events.columns:
        tr = pd.to_numeric(train_events["group_idx"], errors="coerce").dropna().unique()
        te = pd.to_numeric(test_events["group_idx"], errors="coerce").dropna().unique()
        if len(tr) and len(te):
            min_group_idx_gap = float(np.min(np.abs(tr[:, None] - te[None, :])))

    return {
        "dataset_label": dataset_label,
        "train_frac": train_frac,
        "repeat": repeat,
        "split_unit_actual": split_meta.get("split_unit_actual", ""),
        "split_group_col": split_meta.get("split_group_col", ""),
        "split_strategy_detail": split_meta.get("split_strategy_detail", ""),
        "n_train": len(train_events),
        "n_test": len(test_events),
        "n_train_groups": split_meta.get("n_train_groups", np.nan),
        "n_test_groups": split_meta.get("n_test_groups", np.nan),
        "n_purged_groups": split_meta.get("n_purged_groups", 0),
        "sample_id_overlap_count": len(train_sample_ids & test_sample_ids),
        "file_group_overlap_count": len(train_groups & test_groups),
        "exact_mother_xyz_overlap_count": exact_overlap,
        "rounded6_mother_xyz_overlap_count": rounded_overlap,
        "min_group_idx_gap": min_group_idx_gap,
        "baseline_feature_columns": "same_file_same_T_all_cells ctx_*",
        "baseline_model_target_columns": ",".join(TARGET_COLS),
        "evaluation_target_columns": ",".join(EVAL_TARGET_COLS),
    }

In [ ]:
def summarize_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["dataset_label", "train_size", "model_name", "target", "target_display", "target_group"]
    agg_spec = dict(
        n_repeats=("repeat", "nunique"),
        train_fraction_actual_mean=("train_fraction_actual", "mean"),
        n_train_mean=("n_train", "mean"),
        n_test_mean=("n_test", "mean"),
    )
    for key in METRIC_KEYS:
        agg_spec[f"{key}_mean"] = (key, "mean")
        agg_spec[f"{key}_std"] = (key, "std")
    if "n_train_groups" in metrics_df.columns:
        agg_spec["n_train_groups_mean"] = ("n_train_groups", "mean")
    if "n_test_groups" in metrics_df.columns:
        agg_spec["n_test_groups_mean"] = ("n_test_groups", "mean")
    return metrics_df.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()

def has_adjacent_terms(selected_terms: Mapping[str, List[str]]) -> bool:
    return any("_adj_" in str(feat) for feats in selected_terms.values() for feat in feats)

def require_eval_adjacency_if_needed(
    selected_terms_by_label: Mapping[str, Mapping[str, List[str]]],
    eval_features_by_label: Mapping[str, pd.DataFrame],
    eval_adjacency_paths: Mapping[str, str],
    allow_missing_adjacency_zero: bool,
) -> None:
    """Guard against silently zeroing adjacency terms.

    A label merely having an --eval-adjacency path is not sufficient.  The path
    must contain the matrix for the biological stage of the mother cells.  For
    example, a dataset whose mother cells are in the 8-cell stage needs G4.csv,
    while a dataset whose mother cells are in the 12-cell stage needs G5.csv.
    """
    if allow_missing_adjacency_zero:
        return
    problems: List[str] = []
    for label, terms in selected_terms_by_label.items():
        if not has_adjacent_terms(terms):
            continue
        if not eval_adjacency_paths.get(label):
            problems.append(f"{label}: --eval-adjacency was not provided")
            continue
        feats = eval_features_by_label[label]
        if "adjacency_stage_status" not in feats.columns:
            problems.append(f"{label}: feature library is missing the adjacency_stage_status audit column")
            continue
        missing_rows = feats[feats["adjacency_stage_status"].astype(str) == "missing"]
        if not missing_rows.empty:
            stages = sorted(missing_rows["mother_cell_stage_count"].dropna().astype(int).unique().tolist())
            problems.append(
                f"{label}: fixed terms contain _adj_, but mother stages {stages} have no matching adjacency matrix;"
                "pass the adj/ directory and make sure it contains the corresponding G*.csv files"
            )
    if problems:
        raise ValueError(
            "Adjacency feature check failed."
            + " | ".join(problems)
            + ".The mapping rule is G1/G2/G3/G4/G5 -> 4/6/7/8/12 cell stages."
              "If you intentionally want missing adjacency features to be zero-filled, add --allow-missing-adjacency-zero."
        )

In [ ]:
def make_audit_rows(
    source_events: pd.DataFrame,
    source_feature_df: pd.DataFrame,
    eval_events_by_label: Mapping[str, pd.DataFrame],
    eval_features_by_label: Mapping[str, pd.DataFrame],
    selected_terms_by_label: Mapping[str, Mapping[str, List[str]]],
    adjacency_paths_by_label: Mapping[str, str],
    source_adjacency_path: Optional[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    data_rows: List[Dict[str, object]] = []
    all_sets = [("SOURCE_T0_T1_T2", source_events, source_feature_df, source_adjacency_path)]
    for label in eval_events_by_label:
        all_sets.append((label, eval_events_by_label[label], eval_features_by_label[label], adjacency_paths_by_label.get(label)))
    for label, events, feats, adj_path in all_sets:
        feature_cols = feature_value_columns(feats)
        adj_cols = [c for c in feature_cols if "_adj_" in c]
        data_rows.append({
            "dataset_label": label,
            "n_events": len(events),
            "n_files": events["file_name"].nunique() if "file_name" in events else np.nan,
            "transitions": ",".join(sorted(events["transition"].astype(str).unique())),
            "mother_names": ",".join(sorted(events["mother_name"].astype(str).unique())),
            "feature_library_n_cols": len(feature_cols),
            "feature_library_adj_cols": len(adj_cols),
            "mother_cell_stage_counts": ",".join(map(str, sorted(feats.get("mother_cell_stage_count", pd.Series(dtype=int)).dropna().astype(int).unique().tolist()))),
            "adjacency_stage_statuses": ",".join(sorted(feats.get("adjacency_stage_status", pd.Series(dtype=str)).dropna().astype(str).unique().tolist())),
            "adjacency_path": adj_path or "",
            "model_target_columns": ",".join(TARGET_COLS),
            "evaluation_target_columns": ",".join(EVAL_TARGET_COLS),
        })
    data_audit = pd.DataFrame(data_rows)

    feature_rows: List[Dict[str, object]] = []
    for label in eval_events_by_label:
        for baseline_model, notes in [
            ("MotherCopyBaseline", "copies mother_x/mother_y/mother_z for x/y/z mean targets; uses train means for half-diff targets"),
            ("LinearRegression", "same-CSV/same-T all-current-cells baseline; direct raw six-target linear regression"),
            ("RandomForest", "same-CSV/same-T all-current-cells RF; direct raw six-target regression with adaptive small-data RF: n_estimators=300, max_features=0.7, min_samples_leaf≈8% of n_train capped at 5"),
            ("MLP", "same-CSV/same-T all-current-cells MLP; direct raw six-target regression; hidden_layer_sizes=(256,)")
        ]:
            feature_rows.append({
                "dataset_label": label,
                "model_name": baseline_model,
                "input_columns_used": "mother_xyz_copy" if baseline_model == "MotherCopyBaseline" else "baseline_same_time_cell_context_features_<label>.csv ctx_* columns",
                "uses_CellFeatureExtractor_library": False,
                "uses_raw_xyz_only": False,
                "notes": notes,
            })
        for model_name in ["FixedTerm_OLS", "FixedTerm_OLS_binaryAdj", "FixedTerm_OLS_binaryAdj_context"]:
            for target, feats in selected_terms_by_label[label].items():
                feature_rows.append({
                    "dataset_label": label,
                    "model_name": model_name,
                    "target": target,
                    "input_columns_used": ",".join(feats) + (", plus ftctx_* same-CSV/same-T context columns" if model_name.endswith("_context") else ""),
                    "n_terms": len(feats),
                    "n_adjacent_terms": sum("_adj_" in f for f in feats),
                    "uses_CellFeatureExtractor_library": True,
                    "uses_raw_xyz_only": False,
                    "notes": (
                        "original weighted Gi.csv contact strengths" if model_name == "FixedTerm_OLS" else
                        "nonzero Gi.csv contacts binarized to equal weight 1" if model_name == "FixedTerm_OLS_binaryAdj" else
                        "binary Gi.csv contacts plus same-CSV/same-T all-current-cell context coordinates appended to each independent target OLS"
                    ),
                })
    feature_audit = pd.DataFrame(feature_rows)
    return data_audit, feature_audit

def safe_filename(text: object) -> str:
    """Filesystem-safe compact filename component."""
    value = str(text)
    keep = []
    for ch in value:
        if ch.isalnum() or ch in {"-", "_", "."}:
            keep.append(ch)
        else:
            keep.append("_")
    out = "".join(keep).strip("_")
    return out or "unnamed"

In [ ]:
def write_prediction_files_for_split(
    output_dir: Path,
    test_events: pd.DataFrame,
    model_predictions: Mapping[str, np.ndarray],
    dataset_label: str,
    train_size: int,
    repeat: int,
) -> None:
    """Save one folder per test split, with one 6-column prediction CSV per method.

    The per-method CSV files intentionally contain only the six predicted target columns
    to avoid the very large long-format test_predictions_all_methods.csv used before.
    y_true.csv and test_metadata.csv are written once per split for comparison.
    """
    split_dir = (
        output_dir
        / "detailed_split_outputs"
        / "test_predictions_by_split"
        / safe_filename(dataset_label)
        / f"train_size_{int(train_size):03d}"
        / f"repeat_{int(repeat):03d}"
    )
    split_dir.mkdir(parents=True, exist_ok=True)

    test_events[EVAL_TARGET_COLS].reset_index(drop=True).to_csv(
        split_dir / "y_true.csv", index=False, encoding="utf-8-sig"
    )
    meta_cols = [c for c in EVENT_META_COLS + XYZ_COLS if c in test_events.columns]
    test_events[meta_cols].reset_index(drop=True).to_csv(
        split_dir / "test_metadata.csv", index=False, encoding="utf-8-sig"
    )
    for model_name, pred_eval in model_predictions.items():
        pred_eval = np.asarray(pred_eval, dtype=float)
        if pred_eval.shape != (len(test_events), len(EVAL_TARGET_COLS)):
            raise ValueError(
                f"prediction shape mismatch for {model_name}: got {pred_eval.shape}, "
                f"expected {(len(test_events), len(EVAL_TARGET_COLS))}"
            )
        pd.DataFrame(pred_eval, columns=EVAL_TARGET_COLS).to_csv(
            split_dir / f"{safe_filename(model_name)}.csv", index=False, encoding="utf-8-sig"
        )

def make_per_experiment_model_summary(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """A compact per split/model summary built from the macro metric rows."""
    if metrics_df.empty:
        return pd.DataFrame()
    keep_targets = ["macro_mean", "macro_mean_position", "macro_half_absdiff"]
    cols = [
        "dataset_label", "train_size", "train_fraction_actual", "repeat", "model_name",
        "target", "target_display", "target_group", "n_train", "n_test",
    ] + METRIC_KEYS
    return metrics_df.loc[metrics_df["target"].isin(keep_targets), [c for c in cols if c in metrics_df.columns]].copy()

def summarize_fixed_os_coefficients(coef_df: pd.DataFrame) -> pd.DataFrame:
    """Mean/std/min/max of the refitted FixedTerm/Fixed OLS coefficients."""
    if coef_df.empty:
        return pd.DataFrame()
    group_cols = [
        "dataset_label", "train_size", "target", "target_display", "order",
        "feature_name", "is_adjacent_feature",
    ]
    existing = [c for c in group_cols if c in coef_df.columns]
    summary = (
        coef_df.groupby(existing, dropna=False)["coefficient"]
        .agg(["count", "mean", "std", "min", "median", "max"])
        .reset_index()
        .rename(columns={
            "count": "n_fits",
            "mean": "coefficient_mean",
            "std": "coefficient_std",
            "min": "coefficient_min",
            "median": "coefficient_median",
            "max": "coefficient_max",
        })
    )
    return summary

In [ ]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Raw CellData evaluation: FixedTerm variants read regression_equations.csv terms; "
            "FixedTerm has original / binary-adjacency / binary-adjacency-plus-context variants; LinearRegression/RF/MLP baselines use per-event same-CSV/same-T all-current-cells context inputs; train sizes default to every integer 10..88."
        )
    )
    parser.add_argument(
        "--source-cell-data",
        required=True,
        help="T0+T1+T2 raw CellData input, for example CellData/CellData_217.csv, CellData/, or a glob. mainly used for audit outputs.",
    )
    parser.add_argument(
        "--source-adjacency",
        default=None,
        help="options：adjacency matrix input for source. Passing adj/ is recommended; G1-G5 map to 4/6/7/8/12 cell stages respectively.",
    )
    parser.add_argument(
        "--eval-cell-data",
        action="append",
        type=parse_eval_cell_data,
        required=True,
        help="target data, e.g. T3_8_12=CellData_812/；can be passed multiple times.",
    )
    parser.add_argument(
        "--eval-adjacency",
        action="append",
        type=parse_eval_adjacency,
        default=[],
        help="adjacency matrices for target data, recommended format such as T3_7_8=adj/、T4_8_12=adj/; or directly LABEL=adj/.",
    )
    parser.add_argument(
        "--regression-equations",
        default="./regression_equations.csv",
        help="FixedTerm_OLS term input file. Only term names are read; existing coefficients are ignored. Default ./regression_equations.csv.",
    )
    parser.add_argument(
        "--term-equation-dir",
        default=None,
        help="legacy-compatible task_*_equation_coefficients.csv directory. If --regression-equations is also provided and exists, regression_equations.csv takes priority.",
    )
    parser.add_argument(
        "--lineage-json",
        default=None,
        help="options lineage JSON, for example {\"ABal\": [\"ABala\", \"ABalp\"]}；the default is the built-in early lineage map.",
    )
    parser.add_argument(
        "--feature-toggles",
        default=None,
        help=(
            "CellFeatureExtractor feature toggles, comma-separated key=true/false. Default "
            "self_single_dim=true,self_coupled_polynomial=true,adjacent_polynomial=true,"
            "self_other_functions=false,adjacent_other_functions=false"
        ),
    )
    parser.add_argument(
        "--train-sizes",
        type=parse_train_sizes,
        default=parse_train_sizes(DEFAULT_TRAIN_SIZE_RANGE_TEXT),
        help=(
            "train sizes, i.e. how many mother-cell division events are randomly selected as the training set."
            "Supports space- or comma-separated values, for example '5 10 20 30'；also supports 20-100:10."
            "Default 5,10,20,30,40,50,60,70,80,90,100."
        ),
    )
    parser.add_argument("--n-repeats", type=int, default=30, help="number of random split repeats for each train size.")
    parser.add_argument("--mlp-max-iter", type=int, default=120, help="MLP maximum number of iterations.Default 120；hidden_layer_sizes=(256,).")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--fit-intercept", action="store_true", help="FixedTerm_OLS whether FixedTerm_OLS fits an intercept.Default: no intercept.")
    parser.add_argument(
        "--allow-missing-adjacency-zero",
        action="store_true",
        help="If fixed terms contain _adj_ terms but an eval dataset has no adjacency matrix, allow these adjacency features to be treated as zero. Default: raise an error.",
    )
    parser.add_argument("--output-dir", default="raw_cell_feature_fixed_terms_eval_v15")
    parser.add_argument(
        "--no-save-test-predictions",
        action="store_true",
        help="By default, create detailed_split_outputs/test_predictions_by_split/ subfolders for each test split and write one 6-column prediction CSV for each method; use this flag to disable it.",
    )
    parser.add_argument("--no-progress", action="store_true", help="disable tqdm progress bars and keep only key-stage logs.")
    return parser.parse_args()

## 11. Command-Line Arguments and Main Workflow

Definitions are split into smaller cells so that the notebook is easier to read and debug.

In [ ]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Raw CellData evaluation: FixedTerm variants read regression_equations.csv terms; "
            "FixedTerm has original / binary-adjacency / binary-adjacency-plus-context variants; LinearRegression/RF/MLP baselines use per-event same-CSV/same-T all-current-cells context inputs; train sizes default to every integer 10..88."
        )
    )
    parser.add_argument(
        "--source-cell-data",
        required=True,
        help="T0+T1+T2 raw CellData input, for example CellData/CellData_217.csv, CellData/, or a glob. mainly used for audit outputs.",
    )
    parser.add_argument(
        "--source-adjacency",
        default=None,
        help="options：adjacency matrix input for source. Passing adj/ is recommended; G1-G5 map to 4/6/7/8/12 cell stages respectively.",
    )
    parser.add_argument(
        "--eval-cell-data",
        action="append",
        type=parse_eval_cell_data,
        required=True,
        help="target data, e.g. T3_8_12=CellData_812/；can be passed multiple times.",
    )
    parser.add_argument(
        "--eval-adjacency",
        action="append",
        type=parse_eval_adjacency,
        default=[],
        help="adjacency matrices for target data, recommended format such as T3_7_8=adj/、T4_8_12=adj/; or directly LABEL=adj/.",
    )
    parser.add_argument(
        "--regression-equations",
        default="./regression_equations.csv",
        help="FixedTerm_OLS term input file. Only term names are read; existing coefficients are ignored. Default ./regression_equations.csv.",
    )
    parser.add_argument(
        "--term-equation-dir",
        default=None,
        help="legacy-compatible task_*_equation_coefficients.csv directory. If --regression-equations is also provided and exists, regression_equations.csv takes priority.",
    )
    parser.add_argument(
        "--lineage-json",
        default=None,
        help="options lineage JSON, for example {\"ABal\": [\"ABala\", \"ABalp\"]}；the default is the built-in early lineage map.",
    )
    parser.add_argument(
        "--feature-toggles",
        default=None,
        help=(
            "CellFeatureExtractor feature toggles, comma-separated key=true/false. Default "
            "self_single_dim=true,self_coupled_polynomial=true,adjacent_polynomial=true,"
            "self_other_functions=false,adjacent_other_functions=false"
        ),
    )
    parser.add_argument(
        "--train-sizes",
        type=parse_train_sizes,
        default=parse_train_sizes(DEFAULT_TRAIN_SIZE_RANGE_TEXT),
        help=(
            "train sizes, i.e. how many mother-cell division events are randomly selected as the training set."
            "Supports space- or comma-separated values, for example '5 10 20 30'；also supports 20-100:10."
            "Default 5,10,20,30,40,50,60,70,80,90,100."
        ),
    )
    parser.add_argument("--n-repeats", type=int, default=30, help="number of random split repeats for each train size.")
    parser.add_argument("--mlp-max-iter", type=int, default=120, help="MLP maximum number of iterations.Default 120；hidden_layer_sizes=(256,).")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--fit-intercept", action="store_true", help="FixedTerm_OLS whether FixedTerm_OLS fits an intercept.Default: no intercept.")
    parser.add_argument(
        "--allow-missing-adjacency-zero",
        action="store_true",
        help="If fixed terms contain _adj_ terms but an eval dataset has no adjacency matrix, allow these adjacency features to be treated as zero. Default: raise an error.",
    )
    parser.add_argument("--output-dir", default="raw_cell_feature_fixed_terms_eval_v15")
    parser.add_argument(
        "--no-save-test-predictions",
        action="store_true",
        help="By default, create detailed_split_outputs/test_predictions_by_split/ subfolders for each test split and write one 6-column prediction CSV for each method; use this flag to disable it.",
    )
    parser.add_argument("--no-progress", action="store_true", help="disable tqdm progress bars and keep only key-stage logs.")
    return parser.parse_args()

## Notebook workflow: run the pipeline block by block

The original `main()` function is intentionally not used here.  Instead, the training/evaluation pipeline is split into explicit notebook cells so that you can rerun only the part you are checking: configuration, source data, evaluation data, term resolution, experiments, summaries, coefficients, and plotting.

### A. Configuration

Edit this cell first.  It replaces the old command-line `argparse` workflow.  The default settings match the v23 train-size sweep: every integer from 10 to 88.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    source_cell_data="CellData/",
    source_adjacency="adj/",
    eval_cell_data=[
        ("T3_8_12", "CellData_812/"),
        ("T4_12_14", "CellData_1214/"),
    ],
    eval_adjacency=[
        ("T3_8_12", "adj/"),
        ("T4_12_14", "adj/"),
    ],
    regression_equations="./regression_equations.csv",
    term_equation_dir=None,
    lineage_json=None,
    feature_toggles=None,
    train_sizes=parse_train_sizes("10-88"),
    n_repeats=1,
    mlp_max_iter=120,
    random_state=42,
    fit_intercept=False,
    allow_missing_adjacency_zero=False,
    output_dir="raw_cell_feature_fixed_terms_eval_v23_modular_notebook",
    no_save_test_predictions=False,
    no_progress=False,
)

out_dir = Path(args.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

lineage_map = load_lineage_json(args.lineage_json)
feature_toggles = parse_feature_toggles(args.feature_toggles)
eval_adjacency_paths = dict(args.eval_adjacency or [])

def log_stage(message: str) -> None:
    print(message, flush=True)

print("Output directory:", out_dir.resolve())
print("Train sizes:", args.train_sizes[:5], "...", args.train_sizes[-5:])

### B. Build source events

This creates the source mother-to-daughter event table from the source CellData files.  The source data is mainly used for term auditing and feature-library consistency checks.

In [ ]:
log_stage("Build source event table")
source_events = build_dataset_from_cell_data(
    args.source_cell_data,
    dataset_label="SOURCE_T0_T1_T2",
    lineage_map=lineage_map,
)
print(source_events.shape)
source_events.head()

### C. Build source adjacency variants and feature libraries

This builds the original weighted Gi feature library, the binary-adjacency feature library, and the binary-adjacency-plus-context feature library.

In [ ]:
log_stage("Build source adjacency and feature libraries")
source_adj_by_t = discover_adjacency_files(args.source_adjacency)
source_adj_by_t_binary = binarize_adjacency_by_stage(source_adj_by_t)

source_feature_df = build_feature_library_for_events(
    events=source_events,
    adjacency_by_t=source_adj_by_t,
    feature_toggles=feature_toggles,
    prefix="mother_",
)
source_feature_df_binary = build_feature_library_for_events(
    events=source_events,
    adjacency_by_t=source_adj_by_t_binary,
    feature_toggles=feature_toggles,
    prefix="mother_",
)
source_feature_df_binary_context = append_same_time_context_to_fixedterm_features(
    source_feature_df_binary,
    source_events,
)

print("weighted:", source_feature_df.shape)
print("binary:", source_feature_df_binary.shape)
print("binary + context:", source_feature_df_binary_context.shape)

### D. Save source audit files

Run this only when you want to refresh the source CSV outputs.

In [ ]:
source_events.to_csv(out_dir / "source_events_T0_T1_T2.csv", index=False, encoding="utf-8-sig")
source_feature_df.to_csv(out_dir / "source_feature_library_T0_T1_T2.csv", index=False, encoding="utf-8-sig")
source_feature_df_binary.to_csv(out_dir / "source_feature_library_binaryAdj_T0_T1_T2.csv", index=False, encoding="utf-8-sig")
source_feature_df_binary_context.to_csv(out_dir / "source_feature_library_binaryAdj_context_T0_T1_T2.csv", index=False, encoding="utf-8-sig")
audit_event_table(source_events, "source").to_csv(out_dir / "source_event_lineage_audit.csv", index=False, encoding="utf-8-sig")
print("Saved source audit files.")

### E. Define a helper to build one evaluation dataset

This keeps the next cell short.  For each evaluation label, it builds events, weighted features, binary-adjacency features, binary-adjacency-context features, and baseline same-time context inputs.

In [ ]:
def build_one_eval_dataset(label: str, path: str):
    events = build_dataset_from_cell_data(path, dataset_label=label, lineage_map=lineage_map)
    adj_by_t = discover_adjacency_files(eval_adjacency_paths.get(label))
    adj_by_t_binary = binarize_adjacency_by_stage(adj_by_t)

    feature_df = build_feature_library_for_events(
        events=events,
        adjacency_by_t=adj_by_t,
        feature_toggles=feature_toggles,
        prefix="mother_",
    )
    feature_df_binary = build_feature_library_for_events(
        events=events,
        adjacency_by_t=adj_by_t_binary,
        feature_toggles=feature_toggles,
        prefix="mother_",
    )
    feature_df_binary_context = append_same_time_context_to_fixedterm_features(feature_df_binary, events)
    baseline_context = build_same_time_cell_context_features(events)
    return events, feature_df, feature_df_binary, feature_df_binary_context, baseline_context

### F. Build all evaluation datasets

This is separated from source construction so you can rerun only target-data processing when checking label/path issues.

In [ ]:
eval_events_by_label = {}
eval_features_by_label = {}
eval_features_binary_by_label = {}
eval_features_binary_context_by_label = {}
eval_baseline_context_by_label = {}

for label, path in args.eval_cell_data:
    log_stage(f"Build eval dataset {label}: {path}")
    events, feature_df, feature_df_binary, feature_df_binary_context, baseline_context = build_one_eval_dataset(label, path)
    eval_events_by_label[label] = events
    eval_features_by_label[label] = feature_df
    eval_features_binary_by_label[label] = feature_df_binary
    eval_features_binary_context_by_label[label] = feature_df_binary_context
    eval_baseline_context_by_label[label] = baseline_context
    print(label, "events", events.shape, "features", feature_df.shape, "binary+context", feature_df_binary_context.shape)

### G. Save evaluation dataset audits

The context CSVs make it easy to verify that each baseline row uses the same CSV and the same time `T` as the mother cell.

In [ ]:
for label in eval_events_by_label:
    events = eval_events_by_label[label]
    feature_df = eval_features_by_label[label]
    feature_df_binary = eval_features_binary_by_label[label]
    feature_df_binary_context = eval_features_binary_context_by_label[label]
    baseline_context = eval_baseline_context_by_label[label]

    events.to_csv(out_dir / f"eval_events_{label}.csv", index=False, encoding="utf-8-sig")
    feature_df.to_csv(out_dir / f"eval_feature_library_{label}.csv", index=False, encoding="utf-8-sig")
    feature_df_binary.to_csv(out_dir / f"eval_feature_library_binaryAdj_{label}.csv", index=False, encoding="utf-8-sig")
    feature_df_binary_context.to_csv(out_dir / f"eval_feature_library_binaryAdj_context_{label}.csv", index=False, encoding="utf-8-sig")
    baseline_context.to_csv(out_dir / f"baseline_same_time_cell_context_features_{label}.csv", index=False, encoding="utf-8-sig")
    audit_event_table(events, label).to_csv(out_dir / f"eval_event_lineage_audit_{label}.csv", index=False, encoding="utf-8-sig")

print("Saved evaluation audit files.")

### H. Load FixedTerm definitions from `regression_equations.csv`

Only the term names are used.  Coefficients in that file are ignored and are refit independently for each target column on each train split.

In [ ]:
source_term_cols = fixed_term_candidate_columns(source_feature_df)
regression_path = Path(args.regression_equations)

if regression_path.exists():
    selected_terms, term_report = load_terms_from_regression_equations(
        str(regression_path),
        term_library_columns=source_term_cols,
    )
elif args.term_equation_dir:
    selected_terms, term_report = load_terms_from_equation_dir(
        args.term_equation_dir,
        term_library_columns=source_term_cols,
    )
else:
    raise FileNotFoundError(
        f"Could not find regression equations at {args.regression_equations}, and no term_equation_dir was provided."
    )

dropped = term_report.attrs.get("dropped_terms")
if isinstance(dropped, pd.DataFrame) and not dropped.empty:
    dropped.to_csv(out_dir / "dropped_terms_not_in_CellFeatureExtractor_library.csv", index=False, encoding="utf-8-sig")
term_report.to_csv(out_dir / "fixed_terms_by_target_source.csv", index=False, encoding="utf-8-sig")
term_report.head()

### I. Resolve FixedTerm columns against each evaluation feature library

This produces three independent term dictionaries: original weighted adjacency, binary adjacency, and binary adjacency plus same-time context.

In [ ]:
selected_terms_by_label = {}
selected_terms_binary_by_label = {}
selected_terms_binary_context_by_label = {}
eval_term_audits = []

for label, feature_df in eval_features_by_label.items():
    resolved, audit = resolve_terms_against_eval_features(selected_terms, fixed_term_candidate_columns(feature_df))
    selected_terms_by_label[label] = resolved
    audit.insert(0, "dataset_label", label)
    audit.insert(1, "fixedterm_variant", "FixedTerm_OLS")
    eval_term_audits.append(audit)

    resolved_binary, audit_binary = resolve_terms_against_eval_features(
        selected_terms,
        fixed_term_candidate_columns(eval_features_binary_by_label[label]),
    )
    selected_terms_binary_by_label[label] = resolved_binary
    audit_binary.insert(0, "dataset_label", label)
    audit_binary.insert(1, "fixedterm_variant", "FixedTerm_OLS_binaryAdj")
    eval_term_audits.append(audit_binary)

    resolved_context = add_context_terms_to_selected_terms(
        resolved_binary,
        eval_features_binary_context_by_label[label],
    )
    selected_terms_binary_context_by_label[label] = resolved_context
    audit_context = pd.DataFrame([
        {
            "dataset_label": label,
            "fixedterm_variant": "FixedTerm_OLS_binaryAdj_context",
            "target": target,
            "target_display": TARGET_DISPLAY.get(target, target),
            "n_original_or_binary_terms": len(resolved_binary[target]),
            "n_same_time_context_terms_added": len(fixedterm_context_value_columns(eval_features_binary_context_by_label[label])),
            "n_total_terms": len(resolved_context[target]),
        }
        for target in TARGET_COLS
    ])
    eval_term_audits.append(audit_context)

fixed_terms_eval_audit = pd.concat(eval_term_audits, ignore_index=True, sort=False)
fixed_terms_eval_audit.to_csv(out_dir / "fixed_terms_presence_in_eval_feature_libraries.csv", index=False, encoding="utf-8-sig")
fixed_terms_eval_audit.head()

### J. Validate adjacency availability and save high-level audit files

In [ ]:
require_eval_adjacency_if_needed(
    selected_terms_by_label=selected_terms_by_label,
    eval_features_by_label=eval_features_by_label,
    eval_adjacency_paths=eval_adjacency_paths,
    allow_missing_adjacency_zero=args.allow_missing_adjacency_zero,
)

data_audit, feature_audit = make_audit_rows(
    source_events=source_events,
    source_feature_df=source_feature_df,
    eval_events_by_label=eval_events_by_label,
    eval_features_by_label=eval_features_by_label,
    selected_terms_by_label=selected_terms_by_label,
    adjacency_paths_by_label=eval_adjacency_paths,
    source_adjacency_path=args.source_adjacency,
)
data_audit["train_size_mode"] = "random_cell_events_selected_sizes"
data_audit["requested_train_sizes"] = ",".join(map(str, args.train_sizes))

data_audit.to_csv(out_dir / "data_audit.csv", index=False, encoding="utf-8-sig")
feature_audit.to_csv(out_dir / "feature_audit.csv", index=False, encoding="utf-8-sig")

data_audit

### K. Prepare split-level experiment helpers

The next cells replace the old monolithic experiment section.  They isolate split preparation, FixedTerm fitting, baseline fitting, metric collection, and optional prediction saving.

In [ ]:
baseline_names = [
    "MotherCopyBaseline",
    "LinearRegression",
    "RandomForest",
    "MLP",
]

fixedterm_variant_names = [
    "FixedTerm_OLS",
    "FixedTerm_OLS_binaryAdj",
    "FixedTerm_OLS_binaryAdj_context",
]

def prepare_split_data(dataset_label: str, train_size: int, repeat: int):
    events = eval_events_by_label[dataset_label]
    feature_df = eval_features_by_label[dataset_label]
    feature_df_binary = eval_features_binary_by_label[dataset_label]
    feature_df_binary_context = eval_features_binary_context_by_label[dataset_label]

    seed = args.random_state + repeat * 1009 + int(train_size) * 9176
    train_idx, test_idx, split_meta = compute_random_cell_train_test_indices_by_size(
        n=len(events),
        train_size=int(train_size),
        seed=seed,
    )

    split = {
        "dataset_label": dataset_label,
        "train_size": int(train_size),
        "repeat": repeat,
        "seed": seed,
        "train_idx": train_idx,
        "test_idx": test_idx,
        "split_meta": split_meta,
        "events": events,
        "train_events": events.iloc[train_idx].reset_index(drop=True),
        "test_events": events.iloc[test_idx].reset_index(drop=True),
        "train_features": feature_df.iloc[train_idx].reset_index(drop=True),
        "test_features": feature_df.iloc[test_idx].reset_index(drop=True),
        "train_features_binary": feature_df_binary.iloc[train_idx].reset_index(drop=True),
        "test_features_binary": feature_df_binary.iloc[test_idx].reset_index(drop=True),
        "train_features_binary_context": feature_df_binary_context.iloc[train_idx].reset_index(drop=True),
        "test_features_binary_context": feature_df_binary_context.iloc[test_idx].reset_index(drop=True),
    }
    split["y_test_eval"] = split["test_events"][EVAL_TARGET_COLS].reset_index(drop=True)
    return split

In [ ]:
def fit_fixedterm_variants_for_split(split: Mapping[str, object]):
    dataset_label = split["dataset_label"]
    train_size = split["train_size"]
    repeat = split["repeat"]

    fixedterm_jobs = [
        (
            "FixedTerm_OLS",
            selected_terms_by_label[dataset_label],
            split["train_features"],
            split["test_features"],
        ),
        (
            "FixedTerm_OLS_binaryAdj",
            selected_terms_binary_by_label[dataset_label],
            split["train_features_binary"],
            split["test_features_binary"],
        ),
        (
            "FixedTerm_OLS_binaryAdj_context",
            selected_terms_binary_context_by_label[dataset_label],
            split["train_features_binary_context"],
            split["test_features_binary_context"],
        ),
    ]

    model_predictions = {}
    coef_rows = []
    for fixed_model_name, fixed_terms_for_model, fixed_train_features, fixed_test_features in fixedterm_jobs:
        fixed_pred, fixed_coef_rows = fit_fixed_term_ols_predict(
            selected_terms=fixed_terms_for_model,
            train_events=split["train_events"],
            test_events=split["test_events"],
            train_features=fixed_train_features,
            test_features=fixed_test_features,
            fit_intercept=args.fit_intercept,
            dataset_label=dataset_label,
            train_size=int(train_size),
            repeat=repeat,
            split_meta=split["split_meta"],
            model_name=fixed_model_name,
        )
        model_predictions[fixed_model_name] = fixed_pred
        coef_rows.extend(fixed_coef_rows)
    return model_predictions, coef_rows

In [ ]:
def fit_baselines_for_split(split: Mapping[str, object]):
    # Build the same-CSV/same-T context on train+test together only to make a fixed-width design matrix.
    # It does not use daughter coordinates or target values.
    combined_events_for_context = pd.concat(
        [split["train_events"], split["test_events"]],
        axis=0,
        ignore_index=True,
    )
    combined_context = build_same_time_cell_context_features(combined_events_for_context)
    train_context = combined_context.iloc[:len(split["train_events"])].reset_index(drop=True)
    test_context = combined_context.iloc[len(split["train_events"]):].reset_index(drop=True)

    model_predictions = {}
    for model_name in baseline_names:
        model_predictions[model_name] = fit_xyz_baseline_predict(
            model_name=model_name,
            train_events=split["train_events"],
            test_events=split["test_events"],
            random_state=split["seed"],
            mlp_max_iter=args.mlp_max_iter,
            train_context_features=train_context,
            test_context_features=test_context,
        )
    return model_predictions

In [ ]:
def collect_metrics_for_split(split: Mapping[str, object], model_predictions: Mapping[str, np.ndarray]):
    n_train = len(split["train_events"])
    n_test = len(split["test_events"])
    metric_rows_out = []

    for model_name, pred_eval in model_predictions.items():
        metric_rows_out.extend(
            metric_rows(
                y_true=split["y_test_eval"],
                y_pred=pred_eval,
                model_name=model_name,
                dataset_label=split["dataset_label"],
                train_size=int(split["train_size"]),
                repeat=split["repeat"],
                n_train=n_train,
                n_test=n_test,
                split_meta=split["split_meta"],
            )
        )

    geometric_rows_out = geometric_metric_rows_for_split(
        model_predictions=model_predictions,
        test_events=split["test_events"],
        y_true=split["y_test_eval"],
        dataset_label=split["dataset_label"],
        train_size=int(split["train_size"]),
        repeat=split["repeat"],
        n_train=n_train,
        n_test=n_test,
        split_meta=split["split_meta"],
    )
    return metric_rows_out, geometric_rows_out

In [ ]:
def collect_split_audits(split: Mapping[str, object]):
    assignment_rows = make_split_assignment_rows_by_size(
        events=split["events"],
        train_idx=split["train_idx"],
        test_idx=split["test_idx"],
        dataset_label=split["dataset_label"],
        train_size=int(split["train_size"]),
        repeat=split["repeat"],
        split_meta=split["split_meta"],
    )
    leakage_row = make_leakage_audit_row_by_size(
        train_events=split["train_events"],
        test_events=split["test_events"],
        dataset_label=split["dataset_label"],
        train_size=int(split["train_size"]),
        repeat=split["repeat"],
        split_meta=split["split_meta"],
    )
    return assignment_rows, leakage_row


def maybe_save_predictions_for_split(split: Mapping[str, object], model_predictions: Mapping[str, np.ndarray]):
    if args.no_save_test_predictions:
        return
    write_prediction_files_for_split(
        output_dir=out_dir,
        test_events=split["test_events"],
        model_predictions=model_predictions,
        dataset_label=split["dataset_label"],
        train_size=int(split["train_size"]),
        repeat=split["repeat"],
    )

### L. Initialize experiment containers

Run this cell before the experiment loop.  Rerunning it clears previous in-memory results.

In [ ]:
all_metric_rows = []
all_geometric_rows = []
coef_rows_all = []
split_assignment_rows = []
leakage_audit_rows = []
skipped_train_size_rows = []

print("Containers initialized.")

### M. Run the train-size experiment loop

This is the only long-running cell.  Because the work has been broken into helper functions above, the loop itself is now compact and easy to inspect.

In [ ]:
total_model_fits = 0
for _dataset_label, _events in eval_events_by_label.items():
    _n = len(_events)
    total_model_fits += sum(1 for _ts in args.train_sizes if int(_ts) < _n) * args.n_repeats * (len(fixedterm_variant_names) + len(baseline_names))

progress = tqdm(total=total_model_fits, desc="model fits", unit="model", disable=args.no_progress)

for dataset_label, events in eval_events_by_label.items():
    if len(events) < 2:
        raise ValueError(f"{dataset_label} has only {len(events)} events; train/test splitting is impossible.")

    for train_size in args.train_sizes:
        if int(train_size) >= len(events):
            skipped_train_size_rows.append({
                "dataset_label": dataset_label,
                "train_size": int(train_size),
                "n_events": int(len(events)),
                "reason": "train_size >= n_events; no test set would remain",
            })
            continue

        for repeat in range(args.n_repeats):
            progress.set_description_str(f"{dataset_label} train={train_size} repeat={repeat + 1}/{args.n_repeats}")

            split = prepare_split_data(dataset_label, int(train_size), repeat)
            fixed_predictions, fixed_coef_rows = fit_fixedterm_variants_for_split(split)
            baseline_predictions = fit_baselines_for_split(split)
            model_predictions = {**fixed_predictions, **baseline_predictions}
            coef_rows_all.extend(fixed_coef_rows)
            progress.update(len(fixedterm_variant_names) + len(baseline_names))

            assignment_rows, leakage_row = collect_split_audits(split)
            split_assignment_rows.extend(assignment_rows)
            leakage_audit_rows.append(leakage_row)

            metric_rows_out, geometric_rows_out = collect_metrics_for_split(split, model_predictions)
            all_metric_rows.extend(metric_rows_out)
            all_geometric_rows.extend(geometric_rows_out)
            maybe_save_predictions_for_split(split, model_predictions)

progress.close()
print("Finished experiment loop.")
print("Metric rows:", len(all_metric_rows))
print("Geometric rows:", len(all_geometric_rows))

### N. Save scalar metrics and split audit tables

In [ ]:
metrics_df = pd.DataFrame(all_metric_rows)
metrics_df.to_csv(out_dir / "per_repeat_task_metrics.csv", index=False, encoding="utf-8-sig")
metrics_df.to_csv(out_dir / "test_metrics_by_experiment.csv", index=False, encoding="utf-8-sig")

per_experiment_summary = make_per_experiment_model_summary(metrics_df)
if not per_experiment_summary.empty:
    per_experiment_summary.to_csv(out_dir / "per_experiment_model_summary.csv", index=False, encoding="utf-8-sig")

if split_assignment_rows:
    pd.DataFrame(split_assignment_rows).to_csv(out_dir / "split_assignments.csv", index=False, encoding="utf-8-sig")
if leakage_audit_rows:
    pd.DataFrame(leakage_audit_rows).to_csv(out_dir / "leakage_audit_by_split.csv", index=False, encoding="utf-8-sig")
if skipped_train_size_rows:
    pd.DataFrame(skipped_train_size_rows).to_csv(out_dir / "skipped_train_sizes.csv", index=False, encoding="utf-8-sig")

print(metrics_df.shape)
metrics_df.head()

### O. Save scalar and geometric summaries

In [ ]:
summary_df = summarize_metrics(metrics_df)
summary_df.to_csv(out_dir / "summary_metrics_mean_std.csv", index=False, encoding="utf-8-sig")

geometric_df = pd.DataFrame(all_geometric_rows)
if not geometric_df.empty:
    geometric_df.to_csv(out_dir / "split_vector_error_by_split.csv", index=False, encoding="utf-8-sig")

geometric_summary_df = summarize_layer_metrics(
    geometric_df,
    group_cols=["dataset_label", "train_size", "model_name"],
) if not geometric_df.empty else pd.DataFrame()

if not geometric_summary_df.empty:
    geometric_summary_df.to_csv(out_dir / "split_vector_error_summary.csv", index=False, encoding="utf-8-sig")

main88 = summary_df[(summary_df["train_size"] == 88) & (summary_df["target"] == "macro_mean")].copy()
main88.to_csv(out_dir / "main_result_train_size88.csv", index=False, encoding="utf-8-sig")
main20 = summary_df[(summary_df["train_size"] == 20) & (summary_df["target"] == "macro_mean")].copy()
main20.to_csv(out_dir / "main_result_train_size20.csv", index=False, encoding="utf-8-sig")

print("summary_df", summary_df.shape)
print("geometric_summary_df", geometric_summary_df.shape)
summary_df.head()

### P. Save FixedTerm refit coefficients

In [ ]:
if coef_rows_all:
    coef_df = pd.DataFrame(coef_rows_all)
    coef_df.to_csv(out_dir / "fixed_term_refit_coefficients.csv", index=False, encoding="utf-8-sig")
    coef_df.to_csv(out_dir / "fixed_os_refit_coefficients.csv", index=False, encoding="utf-8-sig")
    coef_summary = summarize_fixed_os_coefficients(coef_df)
    if not coef_summary.empty:
        coef_summary.to_csv(out_dir / "fixed_os_refit_coefficients_summary.csv", index=False, encoding="utf-8-sig")
    print("coef_df", coef_df.shape)
else:
    print("No coefficient rows were generated.")

### Q. Save run configuration

In [ ]:
config = vars(args).copy()
config["eval_cell_data"] = [{"label": a, "path": b} for a, b in args.eval_cell_data]
config["eval_adjacency"] = [{"label": a, "path": b} for a, b in (args.eval_adjacency or [])]
config["train_sizes"] = args.train_sizes
config["feature_toggles_resolved"] = feature_toggles
config["lineage_map"] = {k: list(v) for k, v in lineage_map.items()}
config["notebook_workflow_note"] = (
    "This notebook replaces the old monolithic main() function with explicit cells for source data, eval data, term resolution, split experiments, summaries, coefficients, and newplot.py plotting."
)

with open(out_dir / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Finished. Output directory:", out_dir.resolve())

## Plotting with `newplot.py`

The cells below contain the plotting code from `newplot.py`, with the result directory defaulting to the notebook's `out_dir`.  You can edit the plotting range and `SELECTED_MODELS` in the first plotting cell.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 1. Fixed parameters
# ============================================================

try:
    RESULT_DIR = out_dir
except NameError:
    RESULT_DIR = Path("raw_cell_feature_fixed_terms_eval_v28_merged_with_rf_default")

PLOT_OUT_DIR = RESULT_DIR / "plots_newplot"
PLOT_OUT_DIR.mkdir(parents=True, exist_ok=True)

PLOT_TRAIN_SIZE_MIN = 10
PLOT_TRAIN_SIZE_MAX = 88
PLOT_TRAIN_SIZE_TICK_STEP = 5

SELECTED_MODELS = [
    "FixedTerm_OLS_binaryAdj",
    "MotherCopyBaseline",
    "LinearRegression",
    "RandomForest_default",
    "RandomForest",
    "MLP",
]

SCALAR_METRICS = ["r2", "rmse", "mae"]

# Set this to None to plot all targets
# It is recommended to start with the three macro targets for clarity
SELECTED_TARGETS = [
    "macro_mean",
    "macro_mean_position",
    "macro_half_absdiff",
    "x_mean",
    "x_half_absdiff",
    "y_mean",
    "y_half_absdiff",
    "z_mean",
    "z_half_absdiff",
]

In [ ]:
# ============================================================
# 2. Utility functions
# ============================================================

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(x))


def read_csv_required(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found：{path}")
    print("Reading：", path)
    return pd.read_csv(path)


def normalize_model_names(df):
    df = df.copy()
    if "model_name" in df.columns:
        df["model_name"] = df["model_name"].astype(str).str.strip()
    if "target" in df.columns:
        df["target"] = df["target"].astype(str).str.strip()
    if "dataset_label" in df.columns:
        df["dataset_label"] = df["dataset_label"].astype(str).str.strip()
    if "train_size" in df.columns:
        df["train_size"] = pd.to_numeric(df["train_size"], errors="coerce")
    return df


def filter_df(df):
    df = normalize_model_names(df)

    df = df[
        (df["train_size"] >= PLOT_TRAIN_SIZE_MIN)
        & (df["train_size"] <= PLOT_TRAIN_SIZE_MAX)
    ].copy()

    df = df[df["model_name"].isin(SELECTED_MODELS)].copy()

    if SELECTED_TARGETS is not None and "target" in df.columns:
        df = df[df["target"].isin(SELECTED_TARGETS)].copy()

    return df


def model_color_map():
    cmap = plt.get_cmap("tab10")
    return {m: cmap(i % 10) for i, m in enumerate(SELECTED_MODELS)}


COLORS = model_color_map()


def apply_x_axis(ax):
    ax.set_xlim(PLOT_TRAIN_SIZE_MIN - 0.5, PLOT_TRAIN_SIZE_MAX + 0.5)
    ticks = list(range(PLOT_TRAIN_SIZE_MIN, PLOT_TRAIN_SIZE_MAX + 1, PLOT_TRAIN_SIZE_TICK_STEP))
    if PLOT_TRAIN_SIZE_MAX not in ticks:
        ticks.append(PLOT_TRAIN_SIZE_MAX)
    ax.set_xticks(ticks)


def bottom_legend(ax):
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        fontsize=8,
        frameon=True,
    )

In [ ]:
# ============================================================
# 3. Read and check
# ============================================================

summary = read_csv_required(RESULT_DIR / "summary_metrics_mean_std.csv")
per_repeat = read_csv_required(RESULT_DIR / "per_repeat_task_metrics.csv")
split_summary = read_csv_required(RESULT_DIR / "split_vector_error_summary.csv")
split_raw = read_csv_required(RESULT_DIR / "split_vector_error_by_split.csv")

summary = filter_df(summary)
per_repeat = filter_df(per_repeat)
split_summary = normalize_model_names(split_summary)
split_raw = normalize_model_names(split_raw)

split_summary = split_summary[
    (split_summary["train_size"] >= PLOT_TRAIN_SIZE_MIN)
    & (split_summary["train_size"] <= PLOT_TRAIN_SIZE_MAX)
    & (split_summary["model_name"].isin(SELECTED_MODELS))
].copy()

split_raw = split_raw[
    (split_raw["train_size"] >= PLOT_TRAIN_SIZE_MIN)
    & (split_raw["train_size"] <= PLOT_TRAIN_SIZE_MAX)
    & (split_raw["model_name"].isin(SELECTED_MODELS))
].copy()


print("\nsummary models:")
print(summary["model_name"].value_counts())

print("\nper_repeat models:")
print(per_repeat["model_name"].value_counts())

print("\nsplit_summary models:")
print(split_summary["model_name"].value_counts())

print("\nsplit_raw models:")
print(split_raw["model_name"].value_counts())


missing_summary = [m for m in SELECTED_MODELS if m not in set(summary["model_name"])]
missing_per_repeat = [m for m in SELECTED_MODELS if m not in set(per_repeat["model_name"])]

print("\nMissing models in summary：", missing_summary)
print("Missing models in per_repeat：", missing_per_repeat)

if "RandomForest_default" not in set(summary["model_name"]):
    raise RuntimeError("summary_metrics_mean_std.csv does not contain RandomForest_default, so it cannot be drawn in line plots.")

if "RandomForest_default" not in set(per_repeat["model_name"]):
    raise RuntimeError("per_repeat_task_metrics.csv does not contain RandomForest_default, so it cannot be drawn in boxplots.")

In [ ]:
# ============================================================
# 4. Line plots：summary_metrics_mean_std.csv
# ============================================================

def plot_scalar_lines():
    for metric in SCALAR_METRICS:
        mean_col = f"{metric}_mean"
        std_col = f"{metric}_std"

        if mean_col not in summary.columns:
            print(f"Skip {metric}; does not contain {mean_col}")
            continue

        for (dataset_label, target), df_g in summary.groupby(["dataset_label", "target"], dropna=False):
            fig, ax = plt.subplots(figsize=(10, 6))

            actually_plotted = []

            for model in SELECTED_MODELS:
                df_m = df_g[df_g["model_name"] == model].sort_values("train_size")

                if df_m.empty:
                    print(f"[Missing in line plots] dataset={dataset_label}, target={target}, metric={metric}, model={model}")
                    continue

                x = df_m["train_size"].to_numpy(dtype=float)
                y = df_m[mean_col].to_numpy(dtype=float)

                ax.plot(
                    x,
                    y,
                    marker="o",
                    markersize=3.5,
                    linewidth=1.6,
                    color=COLORS[model],
                    label=model,
                )

                if std_col in df_m.columns:
                    s = df_m[std_col].fillna(0.0).to_numpy(dtype=float)
                    ax.fill_between(x, y - s, y + s, color=COLORS[model], alpha=0.12)

                actually_plotted.append(model)

            if not actually_plotted:
                plt.close(fig)
                continue

            ax.set_title(f"{dataset_label} | {target} | {metric} vs train size")
            ax.set_xlabel("Train size")
            ax.set_ylabel(metric)
            apply_x_axis(ax)

            if metric == "r2":
                ax.set_ylim(-1.0, 1.0)

            ax.grid(True, alpha=0.3)
            bottom_legend(ax)
            fig.tight_layout(rect=[0, 0.08, 1, 1])

            out = PLOT_OUT_DIR / f"line_{safe_name(dataset_label)}_{safe_name(target)}_{metric}.png"
            fig.savefig(out, dpi=300)
            plt.close(fig)
            print("saved:", out)

In [ ]:
# ============================================================
# 5. Boxplots：per_repeat_task_metrics.csv
# ============================================================

def plot_scalar_boxes():
    for metric in SCALAR_METRICS:
        if metric not in per_repeat.columns:
            print(f"Skip {metric}; per_repeat does not contain this column")
            continue

        for (dataset_label, target), df_g in per_repeat.groupby(["dataset_label", "target"], dropna=False):
            data = []
            labels = []
            colors = []

            for model in SELECTED_MODELS:
                vals = (
                    df_g[df_g["model_name"] == model][metric]
                    .replace([np.inf, -np.inf], np.nan)
                    .dropna()
                    .to_numpy(dtype=float)
                )

                if len(vals) == 0:
                    print(f"[Missing in boxplots] dataset={dataset_label}, target={target}, metric={metric}, model={model}")
                    continue

                data.append(vals)
                labels.append(model)
                colors.append(COLORS[model])

            if not data:
                continue

            fig, ax = plt.subplots(figsize=(max(10, 1.3 * len(labels)), 6))

            bp = ax.boxplot(
                data,
                labels=labels,
                patch_artist=True,
                showfliers=False,
            )

            for patch, c in zip(bp["boxes"], colors):
                patch.set_facecolor(c)
                patch.set_alpha(0.25)
                patch.set_edgecolor(c)

            for median in bp["medians"]:
                median.set_linewidth(1.8)

            rng = np.random.default_rng(42)
            for i, vals in enumerate(data, start=1):
                jitter = rng.normal(0.0, 0.055, size=len(vals))
                ax.scatter(
                    np.full(len(vals), i) + jitter,
                    vals,
                    s=16,
                    alpha=0.55,
                    color=colors[i - 1],
                    edgecolors="none",
                )

            ax.set_title(
                f"{dataset_label} | {target} | {metric} distribution "
                f"({PLOT_TRAIN_SIZE_MIN}-{PLOT_TRAIN_SIZE_MAX})"
            )
            ax.set_xlabel("Model")
            ax.set_ylabel(metric)

            if metric == "r2":
                ax.set_ylim(-1.0, 1.0)

            ax.grid(True, axis="y", alpha=0.3)
            ax.tick_params(axis="x", rotation=30)

            handles = [
                plt.Line2D([0], [0], marker="s", linestyle="", color=COLORS[m], label=m, markersize=8)
                for m in labels
            ]
            ax.legend(
                handles=handles,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.5),
                ncol=3,
                fontsize=8,
                frameon=True,
            )

            fig.tight_layout(rect=[0, 0.13, 1, 1])

            out = PLOT_OUT_DIR / f"box_{safe_name(dataset_label)}_{safe_name(target)}_{metric}.png"
            fig.savefig(out, dpi=300)
            plt.close(fig)
            print("saved:", out)

In [ ]:
# ============================================================
# 6. split_vector_error Line plots
# ============================================================

def plot_split_vector_lines():
    metric = "split_vector_error"
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    if mean_col not in split_summary.columns:
        print(f"split_summary does not contain {mean_col}")
        print(split_summary.columns)
        return

    for dataset_label, df_g in split_summary.groupby("dataset_label", dropna=False):
        fig, ax = plt.subplots(figsize=(10, 6))

        actually_plotted = []

        for model in SELECTED_MODELS:
            df_m = df_g[df_g["model_name"] == model].sort_values("train_size")

            if df_m.empty:
                print(f"[split Missing in line plots] dataset={dataset_label}, model={model}")
                continue

            x = df_m["train_size"].to_numpy(dtype=float)
            y = df_m[mean_col].to_numpy(dtype=float)

            ax.plot(
                x,
                y,
                marker="o",
                markersize=3.5,
                linewidth=1.6,
                color=COLORS[model],
                label=model,
            )

            if std_col in df_m.columns:
                s = df_m[std_col].fillna(0.0).to_numpy(dtype=float)
                ax.fill_between(x, y - s, y + s, color=COLORS[model], alpha=0.12)

            actually_plotted.append(model)

        if not actually_plotted:
            plt.close(fig)
            continue

        ax.set_title(f"{dataset_label} | split_vector_error vs train size")
        ax.set_xlabel("Train size")
        ax.set_ylabel("split_vector_error")
        apply_x_axis(ax)
        ax.grid(True, alpha=0.3)
        bottom_legend(ax)
        fig.tight_layout(rect=[0, 0.08, 1, 1])

        out = PLOT_OUT_DIR / f"line_{safe_name(dataset_label)}_split_vector_error.png"
        fig.savefig(out, dpi=300)
        plt.close(fig)
        print("saved:", out)

In [ ]:
# ============================================================
# 7. split_vector_error Boxplots
# ============================================================

def plot_split_vector_boxes():
    metric = "split_vector_error"

    if metric not in split_raw.columns:
        print(f"split_raw does not contain {metric}")
        print(split_raw.columns)
        return

    for dataset_label, df_g in split_raw.groupby("dataset_label", dropna=False):
        data = []
        labels = []
        colors = []

        for model in SELECTED_MODELS:
            vals = (
                df_g[df_g["model_name"] == model][metric]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .to_numpy(dtype=float)
            )

            if len(vals) == 0:
                print(f"[split Missing in boxplots] dataset={dataset_label}, model={model}")
                continue

            data.append(vals)
            labels.append(model)
            colors.append(COLORS[model])

        if not data:
            continue

        fig, ax = plt.subplots(figsize=(max(10, 1.3 * len(labels)), 6))

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            showfliers=False,
        )

        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.25)
            patch.set_edgecolor(c)

        rng = np.random.default_rng(42)
        for i, vals in enumerate(data, start=1):
            jitter = rng.normal(0.0, 0.055, size=len(vals))
            ax.scatter(
                np.full(len(vals), i) + jitter,
                vals,
                s=16,
                alpha=0.55,
                color=colors[i - 1],
                edgecolors="none",
            )

        ax.set_title(
            f"{dataset_label} | split_vector_error distribution "
            f"({PLOT_TRAIN_SIZE_MIN}-{PLOT_TRAIN_SIZE_MAX})"
        )
        ax.set_xlabel("Model")
        ax.set_ylabel("split_vector_error")
        ax.grid(True, axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=30)

        handles = [
            plt.Line2D([0], [0], marker="s", linestyle="", color=COLORS[m], label=m, markersize=8)
            for m in labels
        ]
        ax.legend(
            handles=handles,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.5),
            ncol=3,
            fontsize=8,
            frameon=True,
        )

        fig.tight_layout(rect=[0, 0.13, 1, 1])

        out = PLOT_OUT_DIR / f"box_{safe_name(dataset_label)}_split_vector_error.png"
        fig.savefig(out, dpi=300)
        plt.close(fig)
        print("saved:", out)

In [ ]:
# ============================================================
# 8. Run
# ============================================================

plot_scalar_lines()
plot_scalar_boxes()
plot_split_vector_lines()
plot_split_vector_boxes()

print("All figures saved to：", PLOT_OUT_DIR)